# MARST & ST-Transformer V5 — Multi-Dataset Imputation Benchmark (Anti-Cheat v4)
### Datasets: PEMS-BAY, METR-LA (speed) + PEMS04, PEMS08 (flow). Set `DATASET` in the config cell; run once per dataset.
Speed datasets use the zero-as-missing convention; flow datasets are fully observed with synthetic random masking. 80% sparsity, 5 seeds.

Two spatiotemporal-transformer models are trained head-to-head over the same trunk:
**V5** (single `soft_locf` anchor + meta-gate) and **MARST** (Multi-Anchor Residual
Spatiotemporal Transformer — a learned softmax mixture of LOCF / HA / KNN anchors).
The notebook also runs a V5 component **ablation**, a **sparsity-sensitivity** sweep,
an extended **evaluation-metrics** table, and **cross-dataset** aggregation.

All anti-leak invariants hold per dataset: a blind (masked) sensor never sees its own value, no future leakage, train-only stats, no self-loops.

**Anti-cheat audit v4** — every model receives *only the same 20% observed sensor readings*
as input. No model sees the held-out 80% or future time steps.

### Fixes applied (baselines & evaluation)

| # | Fix | Models affected |
|---|-----|-----------------|
| v2-1 | BiLSTM bidirectional→forward | BiLSTM |
| v2-2 | BRITS backward GRU removed | BRITS-lite |
| v2-3 | SAITS/ASTGCN causal attention mask | SAITS-lite, ASTGCN-lite |
| v3-1 | Unified numpy RNG for eval masks | ALL models |
| v3-2 | Eval chunk size = training window (48) | Our models |
| v4-1 | Node embedding added | MLP, SAITS-lite |
| v4-2 | KNN masked-distance (only observed dims) | KNN |
| v4-3 | Graph models: HA fill for unobserved nodes | DCRNN, GWN, ASTGCN |
| v4-5 | Staleness feature (steps-since-last-obs / 48) | V5, MARST |
| v4-6 | Soft LOCF: EMA(0.95) toward HA prior | V5 |
| v4-7 | Masking curriculum 60%→80% over 600 epochs | V5, MARST |
| v4-8 | Larger model: hidden 128, 6 layers, 2000 epochs | V5, MARST |

**v4-1 rationale:** at 80% sparsity, unobserved nodes receive input `[0, 0, sin, cos]`.
Without a node embedding, MLP and SAITS cannot distinguish sensors and collapse to
a global prediction — explaining their worse-than-HA results.

**v4-3 rationale:** graph diffusion was spreading masked zeros into neighbouring nodes,
corrupting the spatial signal. Filling with the HA prior gives a neutral, informative
starting point while the mask feature still tells the model which nodes are real.


In [13]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
print("GPU memory ready.")

GPU memory ready.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import glob
import pickle
import urllib.request
import warnings
import json

warnings.filterwarnings("ignore")

GLOBAL_SEED = 42
torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================================
#  DATASET SWITCH  ->  one of: 'PEMS-BAY', 'METR-LA', 'PEMS04', 'PEMS08'
#  Run the whole notebook once per dataset. Each run writes results_<DATASET>.json;
#  the cross-dataset aggregation cell near the end stitches all saved runs together.
# ============================================================================
DATASET = 'PEMS-BAY'

# --- Shared experiment config (identical across datasets for comparability) ---
SPARSITY      = 0.80      # fraction of sensor readings hidden (blind) at train/eval
BATCH_TIME    = 48
HIDDEN_DIM    = 96
N_LAYERS      = 5
N_HEADS       = 4
DROPOUT       = 0.1
TRAIN_EPOCHS  = 1200
STEPS_PER_DAY = 288       # all four datasets are 5-min sampled -> 288 steps/day
EVAL_SEEDS    = [42, 43, 44, 45, 46]
HUBER_BETA    = 1.0

WINDOW     = 5000         # use first WINDOW timesteps (keeps runs comparable & fast)
TRAIN_END  = 4000
EVAL_START = 4500
EVAL_LEN   = 450
# gap [TRAIN_END:EVAL_START] separates train and eval -> no temporal overlap.

# --- Per-dataset metadata -----------------------------------------------------
#  kind 'speed' : raw value 0 == missing reading (PEMS-BAY / METR-LA convention)
#  kind 'flow'  : .npz [T, N, C]; every entry is real (valid all-ones). Missingness
#                 is purely synthetic via the random mask. Impute channel 0 (flow).
DATASETS = {
    'PEMS-BAY': dict(kind='speed', steps_per_day=288, unit='km/h',
                     data_files=['pems-bay.h5', 'PEMS-BAY.h5', 'pems-bay.csv', 'PEMS-BAY.csv'],
                     adj_files=['adj_mx_bay.pkl', 'adj_mx_pems_bay.pkl'],
                     data_url='https://zenodo.org/records/5146275/files/PEMS-BAY.csv?download=1',
                     adj_url='https://zenodo.org/records/5146275/files/adj_mx_bay.pkl?download=1'),
    'METR-LA':  dict(kind='speed', steps_per_day=288, unit='km/h',
                     data_files=['metr-la.h5', 'METR-LA.h5', 'metr-la.csv', 'METR-LA.csv'],
                     adj_files=['adj_mx.pkl', 'adj_mx_la.pkl', 'adj_mx_metr_la.pkl'],
                     data_url='https://zenodo.org/records/5146275/files/METR-LA.csv?download=1',
                     adj_url='https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/adj_mx.pkl'),
    'PEMS04':   dict(kind='flow', steps_per_day=288, channel=0, unit='veh/5min',
                     data_files=['pems04.npz', 'PEMS04.npz', 'pems04_data.npz'],
                     adj_files=['distance_04.csv', 'PEMS04_distance.csv', 'PEMS04.csv', 'pems04.csv', 'distance.csv'],
                     data_url=None, adj_url=None),
    'PEMS08':   dict(kind='flow', steps_per_day=288, channel=0, unit='veh/5min',
                     data_files=['pems08.npz', 'PEMS08.npz', 'pems08_data.npz'],
                     adj_files=['distance_08.csv', 'PEMS08_distance.csv', 'PEMS08.csv', 'pems08.csv', 'distance.csv'],
                     data_url=None, adj_url=None),
}
assert DATASET in DATASETS, f"Unknown DATASET={DATASET!r}; choose one of {list(DATASETS)}"
CFG = DATASETS[DATASET]
DATASET_NAME = DATASET
STEPS_PER_DAY = CFG['steps_per_day']
VALUE_UNIT = CFG['unit']
VALUE_NAME = 'Speed' if CFG['kind'] == 'speed' else 'Flow'


def find_file(candidates, search_roots=('.', '/kaggle/input', '/kaggle/working')):
    cand_lower = [c.lower() for c in candidates]
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for path in glob.glob(os.path.join(root, '**', '*'), recursive=True):
            if os.path.isfile(path) and os.path.basename(path).lower() in cand_lower:
                return path
    return None


def download_if_missing(url, dest):
    if url is None:
        return None
    if not os.path.exists(dest):
        print(f"Downloading {dest} from {url} ...")
        urllib.request.urlretrieve(url, dest)
    return dest


def load_raw_array():
    """Return value_raw [T, N] float32: speed (km/h) or flow (veh/5min)."""
    if CFG['kind'] == 'speed':
        h5 = find_file([f for f in CFG['data_files'] if f.lower().endswith('.h5')])
        if h5 is not None:
            print(f"  H5: {h5}")
            df = pd.read_hdf(h5)
            return np.nan_to_num(df.values.astype(np.float32), nan=0.0)
        csv = find_file([f for f in CFG['data_files'] if f.lower().endswith('.csv')])
        if csv is None:
            csv = download_if_missing(CFG['data_url'], f'{DATASET}.csv')
        if csv is None:
            raise FileNotFoundError(
                f"{DATASET}: no .h5/.csv found and no usable download URL. "
                f"Add the dataset to your Kaggle inputs (expected one of {CFG['data_files']}).")
        print(f"  CSV (timeseries): {csv}")
        df = pd.read_csv(csv, index_col=0)
        return np.nan_to_num(df.values.astype(np.float32), nan=0.0)
    else:  # flow .npz, shape [T, N, C]
        npz = find_file([f for f in CFG['data_files'] if f.lower().endswith('.npz')])
        if npz is None:
            _present = [os.path.basename(p) for r in ('.', '/kaggle/input', '/kaggle/working')
                        if os.path.isdir(r)
                        for p in glob.glob(os.path.join(r, '**', '*.npz'), recursive=True)]
            raise FileNotFoundError(
                f"{DATASET}: no .npz matching {CFG['data_files']} found. "
                f".npz files present: {_present or 'none'}. "
                f"Add the {DATASET} dataset (.npz + edge-list .csv) to your inputs.")
        print(f"  NPZ: {npz}")
        _npz = np.load(npz)
        _key = 'data' if 'data' in _npz.files else _npz.files[0]
        arr = _npz[_key].astype(np.float32)              # [T, N, C]
        ch = CFG.get('channel', 0)
        print(f"  npz shape={arr.shape}, using channel {ch} (flow)")
        return arr[:, :, ch]                              # [T, N]


def load_adjacency(num_nodes):
    """Return binary adjacency [N, N], diagonal zeroed (no self-loop).

    Anti-leak: zeroing the diagonal guarantees the 1-hop / 2-hop neighbour-mean
    features can never read a blind node's own (masked) value back to itself.
    """
    if CFG['kind'] == 'speed':
        pkl = find_file(CFG['adj_files'])
        if pkl is None:
            pkl = download_if_missing(CFG['adj_url'], CFG['adj_files'][0])
        if pkl is None:
            raise FileNotFoundError(f"{DATASET}: adjacency pickle not found ({CFG['adj_files']}).")
        print(f"  Adj PKL: {pkl}")
        with open(pkl, 'rb') as f:
            obj = pickle.load(f, encoding='latin1')
        adj_mx = obj[2] if isinstance(obj, (list, tuple)) and len(obj) >= 3 else obj
        adj_mx = np.asarray(adj_mx, dtype=np.float32)
        if adj_mx.shape[0] != num_nodes:
            raise ValueError(
                f"Adjacency shape {adj_mx.shape} != data ({num_nodes} nodes). "
                f"Wrong pickle picked up -- delete cached adj_mx*.pkl and re-run.")
        adj = (adj_mx > 0.1).astype(np.float32)
    else:  # flow: build from distance.csv edge list (columns: from, to, cost)
        csv = find_file(CFG['adj_files'])
        if csv is None:
            _csvs = [os.path.basename(p) for r in ('.', '/kaggle/input', '/kaggle/working')
                     if os.path.isdir(r)
                     for p in glob.glob(os.path.join(r, '**', '*.csv'), recursive=True)]
            raise FileNotFoundError(
                f"{DATASET}: edge-list not found ({CFG['adj_files']}). "
                f".csv files present: {_csvs or 'none'}.")
        print(f"  Adj CSV (edge list): {csv}")
        ed = pd.read_csv(csv)
        cols = [str(c).lower() for c in ed.columns]
        fi = cols.index('from') if 'from' in cols else 0
        ti = cols.index('to')   if 'to'   in cols else 1
        f_arr = ed.iloc[:, fi].astype(float).astype(int).values
        t_arr = ed.iloc[:, ti].astype(float).astype(int).values
        adj = np.zeros((num_nodes, num_nodes), dtype=np.float32)
        ok = (f_arr >= 0) & (f_arr < num_nodes) & (t_arr >= 0) & (t_arr < num_nodes)
        for a, b in zip(f_arr[ok], t_arr[ok]):
            adj[a, b] = 1.0
            adj[b, a] = 1.0   # symmetric undirected graph
        if adj.sum() == 0:
            raise ValueError(
                f"{DATASET}: edge list '{csv}' produced 0 valid edges for {num_nodes} nodes. "
                f"Wrong distance.csv (node-id range mismatch)? Use the {DATASET}-specific edge list.")
        _oob = int((~ok).sum())
        if _oob > 0.5 * max(len(f_arr), 1):
            print(f"  WARNING: {_oob}/{len(f_arr)} edges fall outside node range [0,{num_nodes}); "
                  f"is '{csv}' the correct edge list for {DATASET}?")
    np.fill_diagonal(adj, 0)
    return adj


print(f"Device: {device} | Dataset: {DATASET_NAME} ({CFG['kind']}) | Target Sparsity: {SPARSITY*100:.0f}%")


In [ ]:
class STBlock(nn.Module):
    def __init__(self, hidden, n_heads, ff_mult=2, dropout=0.0):
        super().__init__()
        self.temp = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads,
            dim_feedforward=hidden * ff_mult, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)
        self.spat = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=n_heads,
            dim_feedforward=hidden * ff_mult, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)

    def forward(self, h, spatial_pad_mask=None):
        B, N, T, H = h.shape
        cm = torch.triu(torch.full((T, T), float('-inf'), device=h.device), diagonal=1)
        h = self.temp(h.reshape(B * N, T, H), src_mask=cm).reshape(B, N, T, H)
        h = h.permute(0, 2, 1, 3).contiguous().reshape(B * T, N, H)
        if spatial_pad_mask is not None:
            h = self.spat(h, src_key_padding_mask=spatial_pad_mask)
        else:
            h = self.spat(h)
        h = h.reshape(B, T, N, H).permute(0, 2, 1, 3).contiguous()
        return h


class MaskedSTTransformerV5(nn.Module):
    """V5 improvements over V4:
    1. Output base: soft_locf + alpha*residual (was locf). Correction learns
       from a better anchor; smaller residual target for stale positions.
    2. 2-hop neighbor mean (n_mean_2hop) as 9th input feature. At 80%
       sparsity each node has ~1.5 observed 1-hop neighbours; 2-hop gives ~5.
    3. Deeper residual head: hidden -> hidden//2 -> 1 (was linear).
    4. Training: 2000 epochs, batch size 4.
    Input features (9): x, m, t_sin, t_cos, n_mean, locf, staleness, soft_locf, n_mean_2hop
    """
    def __init__(self, num_nodes, adj, node_means_t, node_stds_t,
                 hidden=128, n_heads=4, n_layers=6, max_T=288, dropout=0.1,
                 soft_locf_decay=0.95):
        super().__init__()
        self.num_nodes = num_nodes
        self.hidden = hidden
        self.soft_locf_decay = soft_locf_decay
        self.register_buffer('adj_static', adj)
        self.register_buffer('node_means', node_means_t)
        self.register_buffer('node_stds',  node_stds_t)

        # Precompute row-normalised 2-hop adjacency (no self-loops)
        with torch.no_grad():
            adj_sq = torch.matmul(adj, adj)
            adj_sq = adj_sq * (1 - torch.eye(num_nodes, device=adj.device))
            self.register_buffer('adj_2hop',
                                 adj_sq / (adj_sq.sum(1, keepdim=True) + 1e-6))

        self.in_proj  = nn.Linear(9, hidden)
        self.node_emb = nn.Parameter(torch.randn(num_nodes, hidden) * 0.02)
        self.pos_emb  = nn.Parameter(torch.randn(max_T,     hidden) * 0.02)
        self.blocks   = nn.ModuleList([STBlock(hidden, n_heads, dropout=dropout)
                                       for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(hidden)
        self.meta_gate  = nn.Sequential(
            nn.Linear(hidden + 1, 32), nn.GELU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.residual_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.GELU(),
            nn.Linear(hidden // 2, 1))
        self.last_alpha = None

    def _compute_causal_signals(self, x, m, ha_prior):
        """Causal LOCF, staleness (steps-since-obs/48), soft_locf (EMA toward HA)."""
        B, N, T = x.shape
        locf      = torch.zeros_like(x)
        staleness = torch.zeros_like(x)
        soft_locf = torch.zeros_like(x)
        cur_locf  = torch.zeros(B, N, device=x.device)
        cur_stale = torch.zeros(B, N, device=x.device)
        cur_soft  = torch.zeros(B, N, device=x.device)
        decay = self.soft_locf_decay
        for t in range(T):
            obs_t  = x[:, :, t]
            mask_t = m[:, :, t]
            ha_t   = ha_prior[:, :, t]
            obs    = mask_t > 0.5
            cur_locf  = torch.where(obs, obs_t, cur_locf)
            cur_stale = torch.where(obs, torch.zeros_like(cur_stale), cur_stale + 1.0)
            cur_soft  = torch.where(obs, obs_t,
                                    decay * cur_soft + (1.0 - decay) * ha_t)
            locf     [:, :, t] = cur_locf
            staleness[:, :, t] = cur_stale / 48.0
            soft_locf[:, :, t] = cur_soft
        return locf, staleness, soft_locf

    def forward(self, x, m, t_sin, t_cos, ha_prior):
        B, N, T = x.shape
        mean_v = self.node_means.view(1, -1, 1)
        std_v  = self.node_stds .view(1, -1, 1)

        locf, staleness, soft_locf = self._compute_causal_signals(x, m, ha_prior)
        locf_kmh = locf * std_v + mean_v

        # 1-hop neighbour mean (z-scored)
        adj_deg  = self.adj_static.sum(dim=-1, keepdim=True).view(1, -1, 1)
        n_mean   = (torch.matmul(self.adj_static, locf_kmh) / (adj_deg + 1e-6) - mean_v) / std_v

        # 2-hop neighbour mean (z-scored) -- richer spatial context at high sparsity
        n_mean_2hop = (torch.matmul(self.adj_2hop, locf_kmh) - mean_v) / std_v

        feat = torch.stack([x, m, t_sin, t_cos, n_mean, locf,
                            staleness, soft_locf, n_mean_2hop], dim=-1)
        h = self.in_proj(feat)
        h = h + self.node_emb.view(1, N, 1, self.hidden)
        h = h + self.pos_emb[:T].view(1, 1, T, self.hidden)

        spatial_pad_mask = (m == 0).permute(0, 2, 1).contiguous().view(B * T, N)
        for blk in self.blocks:
            h = blk(h, spatial_pad_mask=spatial_pad_mask)
        h = self.final_norm(h)

        residual  = self.residual_head(h).squeeze(-1)
        adj_m_obs = torch.matmul(self.adj_static, m)
        n_obs     = adj_m_obs / (self.adj_static.sum(dim=-1, keepdim=True) + 1e-8)
        alpha     = self.meta_gate(torch.cat([h, n_obs.unsqueeze(-1)], dim=-1)).squeeze(-1)
        self.last_alpha = alpha.detach()

        # Base: soft_locf (not locf) -- correction needs to close a smaller gap
        return soft_locf + alpha * residual



class MaskedSTTransformerV5Ablate(nn.Module):
    """V5 with per-component toggles (defaults == full V5) for the ablation study.

    Always-on input features : x, m, t_sin, t_cos, locf
    Optional input features   : n_mean (1-hop), staleness, soft_locf, n_mean_2hop
    base : 'soft_locf' (default) or 'locf' -- the anchor the residual corrects.

    Anti-leak: adj/adj_2hop have no self-loops; LOCF/staleness/soft_locf are
    causal; masked positions are zeroed by the caller (x * m_eff). A blind
    sensor's own value never enters its own prediction.
    """
    def __init__(self, num_nodes, adj, node_means_t, node_stds_t,
                 hidden=128, n_heads=4, n_layers=6, max_T=288, dropout=0.1,
                 soft_locf_decay=0.95, use_2hop=True, use_soft_locf=True,
                 use_staleness=True, use_n_mean=True, use_node_emb=True,
                 base='soft_locf'):
        super().__init__()
        self.num_nodes = num_nodes
        self.hidden = hidden
        self.soft_locf_decay = soft_locf_decay
        self.use_2hop = use_2hop
        self.use_soft_locf = use_soft_locf
        self.use_staleness = use_staleness
        self.use_n_mean = use_n_mean
        self.use_node_emb = use_node_emb
        self.base = base
        self.register_buffer('adj_static', adj)
        self.register_buffer('node_means', node_means_t)
        self.register_buffer('node_stds',  node_stds_t)
        with torch.no_grad():
            adj_sq = torch.matmul(adj, adj)
            adj_sq = adj_sq * (1 - torch.eye(num_nodes, device=adj.device))
            self.register_buffer('adj_2hop', adj_sq / (adj_sq.sum(1, keepdim=True) + 1e-6))

        n_feat = 5 + int(use_n_mean) + int(use_staleness) + int(use_soft_locf) + int(use_2hop)
        self.in_proj = nn.Linear(n_feat, hidden)
        self.node_emb = nn.Parameter(torch.randn(num_nodes, hidden) * 0.02)
        self.pos_emb  = nn.Parameter(torch.randn(max_T,     hidden) * 0.02)
        self.blocks   = nn.ModuleList([STBlock(hidden, n_heads, dropout=dropout)
                                       for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(hidden)
        self.meta_gate  = nn.Sequential(
            nn.Linear(hidden + 1, 32), nn.GELU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.residual_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.GELU(),
            nn.Linear(hidden // 2, 1))
        self.last_alpha = None

    def _signals(self, x, m, ha_prior):
        B, N, T = x.shape
        locf = torch.zeros_like(x); stale = torch.zeros_like(x); soft = torch.zeros_like(x)
        cl = torch.zeros(B, N, device=x.device)
        cs = torch.zeros(B, N, device=x.device)
        cf = torch.zeros(B, N, device=x.device)
        d = self.soft_locf_decay
        for t in range(T):
            o = m[:, :, t] > 0.5; xt = x[:, :, t]; ht = ha_prior[:, :, t]
            cl = torch.where(o, xt, cl)
            cs = torch.where(o, torch.zeros_like(cs), cs + 1.0)
            cf = torch.where(o, xt, d * cf + (1.0 - d) * ht)
            locf[:, :, t] = cl; stale[:, :, t] = cs / 48.0; soft[:, :, t] = cf
        return locf, stale, soft

    def forward(self, x, m, t_sin, t_cos, ha_prior):
        B, N, T = x.shape
        mean_v = self.node_means.view(1, -1, 1)
        std_v  = self.node_stds.view(1, -1, 1)
        locf, stale, soft = self._signals(x, m, ha_prior)
        locf_kmh = locf * std_v + mean_v

        feats = [x, m, t_sin, t_cos, locf]
        if self.use_n_mean:
            deg = self.adj_static.sum(-1, keepdim=True).view(1, -1, 1)
            feats.append((torch.matmul(self.adj_static, locf_kmh) / (deg + 1e-6) - mean_v) / std_v)
        if self.use_staleness:
            feats.append(stale)
        if self.use_soft_locf:
            feats.append(soft)
        if self.use_2hop:
            feats.append((torch.matmul(self.adj_2hop, locf_kmh) - mean_v) / std_v)
        feat = torch.stack(feats, dim=-1)

        h = self.in_proj(feat)
        if self.use_node_emb:
            h = h + self.node_emb.view(1, N, 1, self.hidden)
        h = h + self.pos_emb[:T].view(1, 1, T, self.hidden)
        spatial_pad_mask = (m == 0).permute(0, 2, 1).contiguous().view(B * T, N)
        for blk in self.blocks:
            h = blk(h, spatial_pad_mask=spatial_pad_mask)
        h = self.final_norm(h)

        residual  = self.residual_head(h).squeeze(-1)
        adj_m_obs = torch.matmul(self.adj_static, m)
        n_obs     = adj_m_obs / (self.adj_static.sum(-1, keepdim=True) + 1e-8)
        alpha     = self.meta_gate(torch.cat([h, n_obs.unsqueeze(-1)], dim=-1)).squeeze(-1)
        self.last_alpha = alpha.detach()
        anchor = soft if self.base == 'soft_locf' else locf
        return anchor + alpha * residual


class MaskedSTTransformerMARST(nn.Module):
    """MARST — Multi-Anchor Residual Spatiotemporal Transformer.

    Generalises the V3–V5 single-anchor residual gate to a *learned mixture*
    of three causal anchors, computed at every (sensor, time) cell:
      a_LOCF : last observed value at this sensor        (recency)
      a_HA   : per-sensor historical average for ToD     (seasonality)
      a_KNN  : mean of currently-observed graph neighbours (spatial context)
    A small head emits a softmax pi in the 2-simplex over the three anchors;
    the blended anchor is  a = pi_LOCF*a_LOCF + pi_HA*a_HA + pi_KNN*a_KNN.
    The spatiotemporal trunk (same alternating temporal/spatial STBlock as V5,
    causal temporal mask + mask-aware spatial attention) predicts a residual r,
    and the meta-gate alpha (as in V3–V5) blends it onto the anchor:
        y_hat = a + alpha * r.

    All anchors are z-scored. Anti-leak invariants are preserved: the adjacency
    diagonal is zero, so a_KNN never reads a node's own (masked) value, and
    every signal is causal (no future leakage).

    Input features (9): x, m, t_sin, t_cos, a_LOCF, a_HA, a_KNN, staleness, n_mean_2hop
    """
    def __init__(self, num_nodes, adj, node_means_t, node_stds_t,
                 hidden=128, n_heads=4, n_layers=6, max_T=288, dropout=0.1,
                 soft_locf_decay=0.95):
        super().__init__()
        self.num_nodes = num_nodes
        self.hidden = hidden
        self.soft_locf_decay = soft_locf_decay
        self.register_buffer('adj_static', adj)
        self.register_buffer('node_means', node_means_t)
        self.register_buffer('node_stds',  node_stds_t)

        # Precompute row-normalised 2-hop adjacency (no self-loops)
        with torch.no_grad():
            adj_sq = torch.matmul(adj, adj)
            adj_sq = adj_sq * (1 - torch.eye(num_nodes, device=adj.device))
            self.register_buffer('adj_2hop',
                                 adj_sq / (adj_sq.sum(1, keepdim=True) + 1e-6))

        self.in_proj  = nn.Linear(9, hidden)
        self.node_emb = nn.Parameter(torch.randn(num_nodes, hidden) * 0.02)
        self.pos_emb  = nn.Parameter(torch.randn(max_T,     hidden) * 0.02)
        self.blocks   = nn.ModuleList([STBlock(hidden, n_heads, dropout=dropout)
                                       for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(hidden)

        # Multi-anchor mixing head: trunk state -> 3 logits -> softmax(pi)
        self.anchor_gate = nn.Sequential(
            nn.Linear(hidden, 32), nn.GELU(),
            nn.Linear(32, 3))
        # Meta-gate and residual head: identical to V5
        self.meta_gate  = nn.Sequential(
            nn.Linear(hidden + 1, 32), nn.GELU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.residual_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.GELU(),
            nn.Linear(hidden // 2, 1))
        self.last_alpha = None
        self.last_pi    = None   # [.,3] anchor mix weights (diagnostic)

    def _compute_causal_signals(self, x, m, ha_prior):
        """Causal LOCF, staleness (steps-since-obs/48), soft_locf (EMA toward HA).
        Kept identical to V5 so shared plotting helpers work unchanged."""
        B, N, T = x.shape
        locf      = torch.zeros_like(x)
        staleness = torch.zeros_like(x)
        soft_locf = torch.zeros_like(x)
        cur_locf  = torch.zeros(B, N, device=x.device)
        cur_stale = torch.zeros(B, N, device=x.device)
        cur_soft  = torch.zeros(B, N, device=x.device)
        decay = self.soft_locf_decay
        for t in range(T):
            obs_t  = x[:, :, t]
            mask_t = m[:, :, t]
            ha_t   = ha_prior[:, :, t]
            obs    = mask_t > 0.5
            cur_locf  = torch.where(obs, obs_t, cur_locf)
            cur_stale = torch.where(obs, torch.zeros_like(cur_stale), cur_stale + 1.0)
            cur_soft  = torch.where(obs, obs_t,
                                    decay * cur_soft + (1.0 - decay) * ha_t)
            locf     [:, :, t] = cur_locf
            staleness[:, :, t] = cur_stale / 48.0
            soft_locf[:, :, t] = cur_soft
        return locf, staleness, soft_locf

    def _knn_anchor(self, x, m, ha_prior):
        """a_KNN: masked mean of currently-observed 1-hop neighbours, z-scored by
        the target node. Where no neighbour is observed, fall back to the HA prior.
        Anti-leak: adj diagonal is zero, so node i never reads its own value."""
        mean_v = self.node_means.view(1, -1, 1)
        std_v  = self.node_stds .view(1, -1, 1)
        x_kmh  = x * std_v + mean_v                          # de-z-score per node
        num    = torch.matmul(self.adj_static, x_kmh * m)    # sum observed nbr kmh
        den    = torch.matmul(self.adj_static, m)            # count observed nbrs
        knn_z  = (num / (den + 1e-6) - mean_v) / std_v       # z-score by target node
        return torch.where(den > 0, knn_z, ha_prior)         # fallback: seasonality

    def forward(self, x, m, t_sin, t_cos, ha_prior):
        B, N, T = x.shape
        mean_v = self.node_means.view(1, -1, 1)
        std_v  = self.node_stds .view(1, -1, 1)

        locf, staleness, soft_locf = self._compute_causal_signals(x, m, ha_prior)
        locf_kmh = locf * std_v + mean_v

        # Three causal anchors (all z-scored)
        a_locf = locf                                    # recency
        a_ha   = ha_prior                                # seasonality
        a_knn  = self._knn_anchor(x, m, ha_prior)        # spatial (observed nbrs)

        # 2-hop neighbour mean of LOCF (z-scored) -- extra spatial context feature
        n_mean_2hop = (torch.matmul(self.adj_2hop, locf_kmh) - mean_v) / std_v

        feat = torch.stack([x, m, t_sin, t_cos, a_locf, a_ha, a_knn,
                            staleness, n_mean_2hop], dim=-1)
        h = self.in_proj(feat)
        h = h + self.node_emb.view(1, N, 1, self.hidden)
        h = h + self.pos_emb[:T].view(1, 1, T, self.hidden)

        spatial_pad_mask = (m == 0).permute(0, 2, 1).contiguous().view(B * T, N)
        for blk in self.blocks:
            h = blk(h, spatial_pad_mask=spatial_pad_mask)
        h = self.final_norm(h)

        # Multi-anchor mixture: pi in the 2-simplex over [LOCF, HA, KNN]
        pi      = torch.softmax(self.anchor_gate(h), dim=-1)     # [B,N,T,3]
        anchors = torch.stack([a_locf, a_ha, a_knn], dim=-1)     # [B,N,T,3]
        blended = (pi * anchors).sum(dim=-1)                     # [B,N,T]
        self.last_pi = pi.detach().reshape(-1, 3)

        # Residual correction gated by alpha (same meta-gate as V5)
        residual  = self.residual_head(h).squeeze(-1)
        adj_m_obs = torch.matmul(self.adj_static, m)
        n_obs     = adj_m_obs / (self.adj_static.sum(dim=-1, keepdim=True) + 1e-8)
        alpha     = self.meta_gate(torch.cat([h, n_obs.unsqueeze(-1)], dim=-1)).squeeze(-1)
        self.last_alpha = alpha.detach()

        return blended + alpha * residual


In [ ]:
print(f"Loading data for {DATASET_NAME} ...")
value_raw = load_raw_array()[:WINDOW]
NUM_NODES = value_raw.shape[1]
print(f"  Detected: T={value_raw.shape[0]}, N={NUM_NODES}, kind={CFG['kind']}")

# Validity mask. Speed datasets: 0 == missing. Flow datasets: fully observed.
if CFG['kind'] == 'speed':
    valid_raw = (value_raw > 0).astype(np.float32)
else:
    valid_raw = np.ones_like(value_raw, dtype=np.float32)
print(f"  Valid fraction: {valid_raw.mean():.3f}")

# Legacy variable name kept so all downstream cells work unchanged.
speed_raw = value_raw

# Clamp range for de-normalised predictions/targets (speed=km/h, flow=veh/5min).
if CFG['kind'] == 'speed':
    CLAMP_LO, CLAMP_HI = 0.0, 120.0
else:
    CLAMP_LO, CLAMP_HI = 0.0, float(value_raw.max()) * 1.5
print(f"  Clamp range: [{CLAMP_LO:.1f}, {CLAMP_HI:.1f}]")

# Error-regime buckets and Hit tolerances (dataset-aware).
# Speed: original km/h values preserved exactly. Flow: derived from data.
if CFG['kind'] == 'speed':
    REGIME_EDGES = (30.0, 60.0)        # km/h: slow / medium / fast traffic
    HIT_TOL      = (5.0, 10.0)         # km/h accuracy tolerances
else:
    _vt = speed_raw[:TRAIN_END][valid_raw[:TRAIN_END] > 0]
    REGIME_EDGES = tuple(float(np.round(p)) for p in np.percentile(_vt, [33, 67]))
    _sc = float(np.median(_vt))
    HIT_TOL = (max(1.0, float(np.round(0.05 * _sc))),
               max(2.0, float(np.round(0.10 * _sc))))
HIT_EPS_MAX = HIT_TOL[1] * 2.0        # x-range for the hit-rate curve
print(f"  Regime edges: {REGIME_EDGES} {VALUE_UNIT} | Hit tol: {HIT_TOL} {VALUE_UNIT}")

# Per-node mean/std over VALID TRAIN entries only (no eval-window leakage).
node_means = np.zeros(NUM_NODES, dtype=np.float32)
node_stds = np.ones(NUM_NODES, dtype=np.float32)
for n in range(NUM_NODES):
    vals = speed_raw[:TRAIN_END, n][valid_raw[:TRAIN_END, n] > 0]
    if len(vals) > 0:
        node_means[n] = vals.mean()
        node_stds[n] = max(float(vals.std()), 1e-3) + 1e-8

speed_norm = (speed_raw - node_means) / node_stds

# HA prior per (node, time-of-day) over VALID TRAIN entries only.
slot_idx = np.arange(len(speed_norm)) % STEPS_PER_DAY
tod_mean = np.zeros((NUM_NODES, STEPS_PER_DAY), dtype=np.float32)
for s in range(STEPS_PER_DAY):
    sel = slot_idx[:TRAIN_END] == s
    sub_data = speed_norm[:TRAIN_END][sel]
    sub_valid = valid_raw[:TRAIN_END][sel]
    sums = (sub_data * sub_valid).sum(axis=0)
    cnts = sub_valid.sum(axis=0) + 1e-8
    tod_mean[:, s] = sums / cnts

ha_prior = torch.tensor(tod_mean[:, slot_idx].T, dtype=torch.float32).to(device)
speed_gpu = torch.tensor(speed_norm, dtype=torch.float32).to(device)
valid_gpu = torch.tensor(valid_raw, dtype=torch.float32).to(device)
node_means_t = torch.tensor(node_means, dtype=torch.float32).to(device)
node_stds_t = torch.tensor(node_stds, dtype=torch.float32).to(device)

adj = load_adjacency(NUM_NODES)
D = np.diag(1.0 / np.sqrt(adj.sum(axis=1) + 1e-8))
adj_norm = D @ adj @ D
A_t = torch.tensor(adj_norm, dtype=torch.float32).to(device)

print(f"Data and adjacency ready. NUM_NODES={NUM_NODES}, avg degree={adj.sum(1).mean():.2f}, edges={int(adj.sum())}")


## Models (Fair, Causal Evaluation)

All models receive **only the 20% observed sensor readings** (`x * m_eff`) as input.
No model has access to the held-out 80% or to future time steps.
Models marked **[fixed]** had future-leakage bugs that have now been corrected.

**Baselines**

| # | Model | Family | Graph? | Causal? |
|---|-------|--------|--------|---------|
| 1 | Historical Average (HA) | Statistical | No | ✓ |
| 2 | LOCF | Statistical | No | ✓ |
| 3 | Global Mean | Statistical | No | ✓ |
| 4 | Ridge Regression | Linear | No | ✓ |
| 5 | KNN Imputer (spatial, train-set neighbors only) | Non-param | No | ✓ |
| 6 | MLP | Neural | No | ✓ |
| 7 | LSTM | RNN | No | ✓ |
| 8 | BiLSTM → 2L-LSTM **[fixed]** | RNN | No | ✓ |
| 9 | GRU | RNN | No | ✓ |
| 10 | TCN (causal dilated) | Conv | No | ✓ |
| 11 | SAITS-lite **[fixed]** | Transformer | No | ✓ |
| 12 | BRITS-lite **[fixed]** | RNN+decay | No | ✓ |
| 13 | DCRNN-lite | GNN+RNN | Yes | ✓ |
| 14 | ASTGCN-lite **[fixed]** | GNN+Attn | Yes | ✓ |
| 15 | GWN-lite | GNN+TCN | Yes | ✓ |

**Our models** (shown as `18.` / `19.` in the results table; the earlier V3/V4 variants have been removed)

| # | Model | Anchor | Graph? | Causal? |
|---|-------|--------|--------|---------|
| 18 | **MaskedSTTransformerV5** | single `soft_locf` + meta-gate | Yes | ✓ |
| 19 | **MARST** (ours) | multi-anchor LOCF/HA/KNN softmax mix + meta-gate | Yes | ✓ |


In [ ]:
import numpy as np, torch

RESULTS = {}
EXT_PRED = {}   # {label: (pred_kmh_flat, true_kmh_flat)} held-out points over EVAL_SEEDS

# ── Shared mask generator (ALL models must use this) ──────────────────────
# Uses numpy default_rng so masks are identical regardless of torch state.
def make_eval_mask_np(seed, EL, N, sparsity=None):
    """Return bool mask [EL, N] — True = observed (20%), False = held-out (80%)."""
    rng = np.random.default_rng(seed)
    sparsity = SPARSITY if sparsity is None else sparsity
    mask = (rng.random((EL, N)) > sparsity).astype(np.float32)  # [T, N]
    # Guarantee >=1 observed sensor per timestep: an all-blind step makes the
    # spatial attention key-padding mask all-True -> softmax(-inf) -> NaN that
    # then spreads across time. At normal sparsity (large N) this never fires,
    # so existing results and per-seed reproducibility are unchanged.
    empty = mask.sum(axis=1) == 0
    if empty.any():
        rows = np.where(empty)[0]
        mask[rows, rng.integers(0, N, size=rows.size)] = 1.0
    return mask

# ── Shared eval helper (numpy, for non-graph models) ──────────────────────
def eval_fn(pred_fn, label):
    ES, EL = 4500, 450
    x_ev = speed_norm[ES:ES+EL]   # [T,N]
    v_ev = valid_raw [ES:ES+EL]
    maes = []; _pp = []; _tt = []
    for seed in EVAL_SEEDS:
        m_ev = make_eval_mask_np(seed, EL, NUM_NODES)  # [T,N]
        sm   = (m_ev == 0) & (v_ev > 0)
        if not sm.any(): continue
        idx  = np.arange(ES, ES+EL)
        p    = np.clip(pred_fn(x_ev.T, v_ev.T, m_ev.T, idx), CLAMP_LO, CLAMP_HI)
        t    = np.clip(x_ev.T * node_stds[:,None] + node_means[:,None], CLAMP_LO, CLAMP_HI)
        maes.append(np.abs(p[sm.T] - t[sm.T]).mean())
        _pp.append(p[sm.T]); _tt.append(t[sm.T])
    arr = np.array(maes)
    print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
    RESULTS[label] = arr
    EXT_PRED[label] = (np.concatenate(_pp), np.concatenate(_tt))

# 1. Historical Average
eval_fn(lambda xn,vn,mn,idx: tod_mean[:,idx%STEPS_PER_DAY]*node_stds[:,None]+node_means[:,None],
        "1. Historical Average (HA)")

# 2. LOCF
def locf(xn, vn, mn, idx):
    N, T = xn.shape
    out  = tod_mean[:,idx%STEPS_PER_DAY]*node_stds[:,None]+node_means[:,None]
    last = out[:,0].copy()
    for t in range(T):
        obs  = (mn[:,t]>0)&(vn[:,t]>0)
        last = np.where(obs, xn[:,t]*node_stds+node_means, last)
        out[:,t] = last
    return out
eval_fn(locf, "2. LOCF (Last-Obs Carried Forward)")

# 3. Global mean
eval_fn(lambda xn,vn,mn,idx: np.broadcast_to(node_means[:,None],xn.shape).copy(),
        "3. Global Mean (per-node train mean)")


In [18]:
from sklearn.linear_model import Ridge
import warnings; warnings.filterwarnings("ignore")

# 4. Ridge Regression (unchanged)
print("Fitting Ridge regressors...")
ridge_models = []
for n in range(NUM_NODES):
    si  = slot_idx[:TRAIN_END]
    Xtr = np.column_stack([np.sin(2*np.pi*si/STEPS_PER_DAY),
                           np.cos(2*np.pi*si/STEPS_PER_DAY),
                           tod_mean[n, si]])
    ytr = speed_norm[:TRAIN_END, n]
    mk  = valid_raw[:TRAIN_END, n] > 0
    clf = Ridge(alpha=1.0); clf.fit(Xtr[mk], ytr[mk])
    ridge_models.append(clf)

def ridge(xn, vn, mn, idx):
    N, T = xn.shape; out = np.zeros((N,T),dtype=np.float32)
    for n in range(N):
        Xn = np.column_stack([np.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY),
                              np.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY),
                              tod_mean[n, idx%STEPS_PER_DAY]])
        out[n] = ridge_models[n].predict(Xn)*node_stds[n]+node_means[n]
    return out
eval_fn(ridge, "4. Node-wise Ridge Regression")

# 5. KNN Imputer — causal spatial imputation with masked Euclidean distance.
#
# Why the previous fix failed: at 80% sparsity, a 325-dim query has ~260 zeros
# (unobserved sensors set to 0). Standard Euclidean distance treats those zeros
# as signal, so neighbours are found based on the zero pattern, not speed values.
#
# Fix: compute distance only over the ~65 observed dimensions, normalised by
# the number of shared observed sensors. This is the standard 'missing-value
# aware' nearest-neighbour approach used in the imputation literature.
# Causal: each timestep is treated independently (spatial, not temporal KNN).
print("Fitting KNN (masked-distance spatial imputer)...")
tr_norm = speed_norm[:TRAIN_END].copy()           # [T_tr, N] z-scored
tr_valid = (valid_raw[:TRAIN_END] > 0).astype(np.float32)  # [T_tr, N]
# Keep only training rows with >=10% valid sensors
enough    = tr_valid.mean(axis=1) >= 0.10
tr_ref    = tr_norm[enough]          # [M, N]
tv_ref    = tr_valid[enough]         # [M, N]  1=valid, 0=missing
K_NEIGH   = 5

def knn_masked(xn, vn, mn, idx):
    """Causal spatial KNN with masked Euclidean distance.
    xn,vn,mn: [N,T] numpy arrays (z-scored / valid / observed-mask).
    Returns [N,T] in km/h.
    Each timestep is solved independently — no future used.
    """
    N, T = xn.shape
    out = (tod_mean[:, idx % STEPS_PER_DAY] * node_stds[:, None]
           + node_means[:, None]).copy()   # HA fallback [N,T]
    for t in range(T):
        obs = (mn[:, t] > 0) & (vn[:, t] > 0)   # [N] bool — truly observed
        if obs.sum() < 2:
            continue   # too few observed sensors — keep HA
        q_vals = xn[:, t]          # [N] z-scored query row

        # Masked squared distance: average over shared valid dimensions only
        # shared[m,n] = 1 if both query and ref row m have sensor n valid
        shared = tv_ref * obs.astype(np.float32)  # [M, N]
        n_shared = shared.sum(axis=1) + 1e-8      # [M]
        diff = (tr_ref - q_vals) * shared         # [M, N] — zero out unshared
        dist = (diff ** 2).sum(axis=1) / n_shared  # [M] mean-sq dist over shared

        # k nearest neighbours
        knn_idx = np.argpartition(dist, K_NEIGH)[:K_NEIGH]  # [K]
        neigh_v  = tr_ref[knn_idx]    # [K, N] z-scored
        neigh_ok = tv_ref[knn_idx]    # [K, N] validity

        # Fill: observed sensors keep their value; missing → weighted avg of neighbours
        neigh_sum = (neigh_v * neigh_ok).sum(0)     # [N]
        neigh_cnt = neigh_ok.sum(0) + 1e-8          # [N]
        imputed   = neigh_sum / neigh_cnt            # [N] z-scored
        row_norm  = np.where(obs, q_vals, imputed)   # [N] z-scored
        out[:, t] = np.clip(row_norm * node_stds + node_means, CLAMP_LO, CLAMP_HI)
    return out

eval_fn(knn_masked, "5. KNN Imputer (k=5, masked-dist)")


Fitting Ridge regressors...
4. Node-wise Ridge Regression                         MAE: 3.1204 +/- 0.0051
Fitting KNN (masked-distance spatial imputer)...
5. KNN Imputer (k=5, masked-dist)                     MAE: 2.1841 +/- 0.0143


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

NE, NH, NLR, NBT = 1200, 64, 1e-3, 48

def train_nw(cls, label, **kw):
    net = cls(4, NH, **kw).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=NLR)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NE)
    xg, vg = speed_gpu[:TRAIN_END], valid_gpu[:TRAIN_END]
    T_tr, N = xg.shape
    for ep in range(1, NE+1):
        net.train()
        t0 = np.random.randint(0, T_tr-NBT)
        xb, vb = xg[t0:t0+NBT], vg[t0:t0+NBT]
        idx = torch.arange(t0, t0+NBT, device=device)
        s   = torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
        c   = torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
        mb  = (torch.rand(NBT, N, device=device) > SPARSITY).float()
        me  = mb * vb; xi = xb * me
        feat = torch.stack([xi.T, me.T,
            s.unsqueeze(0).expand(N, -1),
            c.unsqueeze(0).expand(N, -1)], dim=-1)   # [N,T,4]
        pred = net(feat).squeeze(-1)                  # [N,T]
        lm = (mb.T == 0) & (vb.T > 0)
        if not lm.any(): continue
        loss = F.smooth_l1_loss(pred[lm], xb.T[lm])  # z-scored targets
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
        opt.step(); sch.step()
    print(f"  {label} trained."); return net

def eval_nw(net, label):
    """Uses make_eval_mask_np — identical masks across all models."""
    net.eval()
    ES, EL = 4500, 450
    xev = speed_gpu[ES:ES+EL]; vev = valid_gpu[ES:ES+EL]
    idx = torch.arange(ES, ES+EL, device=device)
    s = torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
    c = torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY)
    st  = torch.tensor(node_stds,  device=device)
    mn_ = torch.tensor(node_means, device=device)
    maes = []; _pp = []; _tt = []
    with torch.no_grad():
        for seed in EVAL_SEEDS:
            m_ev = torch.tensor(
                make_eval_mask_np(seed, EL, NUM_NODES), device=device)  # [T,N]
            me   = m_ev * vev
            feat = torch.stack([
                (xev * me).T, me.T,
                s.unsqueeze(0).expand(NUM_NODES, -1),
                c.unsqueeze(0).expand(NUM_NODES, -1)], dim=-1)  # [N,T,4]
            pred = net(feat).squeeze(-1)               # [N,T]
            pk   = (pred * st[:, None] + mn_[:, None]).clamp(CLAMP_LO, CLAMP_HI)
            tk   = (xev.T * st[:, None] + mn_[:, None]).clamp(CLAMP_LO, CLAMP_HI)
            sm   = (m_ev.T == 0) & (vev.T > 0)
            maes.append(torch.abs(pk[sm] - tk[sm]).mean().item())
            _pp.append(pk[sm].cpu().numpy()); _tt.append(tk[sm].cpu().numpy())
    arr = np.array(maes)
    print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
    RESULTS[label] = arr
    EXT_PRED[label] = (np.concatenate(_pp), np.concatenate(_tt))

# ── MLP with node embedding ───────────────────────────────────────────────
# ROOT CAUSE FIX: at 80% sparsity, an unobserved node's input is [0,0,sin,cos]
# — indistinguishable from any other unobserved node at the same timestep.
# Without a node identity, the MLP predicts the same value for all missing
# nodes, behaving like a global mean. Adding a learned node embedding gives
# each sensor a unique fingerprint even when its value is masked out.
class MLP(nn.Module):
    def __init__(self, F, H, n_nodes=325, **k):
        super().__init__()
        self.node_emb = nn.Embedding(n_nodes, H)
        self.proj = nn.Linear(F, H)
        self.net = nn.Sequential(
            nn.LayerNorm(H * 2),
            nn.Linear(H * 2, H * 2), nn.GELU(),
            nn.LayerNorm(H * 2),
            nn.Linear(H * 2, H),     nn.GELU(),
            nn.Linear(H, 1))
        self._n = n_nodes

    def forward(self, x):   # x: [N, T, F]
        N, T, _ = x.shape
        nids = torch.arange(N, device=x.device)
        ne   = self.node_emb(nids).unsqueeze(1).expand(N, T, -1)  # [N,T,H]
        hf   = self.proj(x)                                        # [N,T,H]
        return self.net(torch.cat([hf, ne], dim=-1))               # [N,T,1]

class LSTM(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__(); self.r=nn.LSTM(F,H,batch_first=True); self.h=nn.Linear(H,1)
    def forward(self,x): o,_=self.r(x); return self.h(o)

# FIXED (v2): bidirectional=True saw future; replaced with 2-layer forward LSTM.
class BiLSTM(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__()
        self.r=nn.LSTM(F, H, num_layers=2, batch_first=True, dropout=0.1)
        self.h=nn.Linear(H,1)
    def forward(self,x): o,_=self.r(x); return self.h(o)

class GRUNet(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__(); self.r=nn.GRU(F,H,batch_first=True); self.h=nn.Linear(H,1)
    def forward(self,x): o,_=self.r(x); return self.h(o)

eval_nw(train_nw(MLP,    "MLP",    n_nodes=NUM_NODES), "6.  MLP (per-node + node-emb)")
eval_nw(train_nw(LSTM,   "LSTM"),                      "7.  LSTM (per-node)")
eval_nw(train_nw(BiLSTM, "BiLSTM"),                    "8.  BiLSTM->2L-LSTM (causal, fixed)")
eval_nw(train_nw(GRUNet, "GRU"),                       "9.  GRU (per-node)")


In [20]:
# 10. TCN — causal dilated convolutions: no fix needed.
class CausalConv(nn.Module):
    def __init__(self,c,k,d):
        super().__init__(); self.p=(k-1)*d; self.c=nn.Conv1d(c,c,k,dilation=d)
    def forward(self,x): return self.c(F.pad(x,(self.p,0)))

class TCNet(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__()
        self.proj=nn.Linear(F,H)
        self.convs=nn.ModuleList([CausalConv(H,3,2**i) for i in range(4)])
        self.norms=nn.ModuleList([nn.LayerNorm(H) for _ in range(4)])
        self.head=nn.Linear(H,1)
    def forward(self,x):
        h=self.proj(x).permute(0,2,1)
        for cv,nm in zip(self.convs,self.norms):
            r=h; h=F.gelu(nm(cv(h).permute(0,2,1))).permute(0,2,1)+r
        return self.head(h.permute(0,2,1))
eval_nw(train_nw(TCNet,"TCN"), "10. TCN (causal dilated, per-node)")

# 11. SAITS-lite — FIXED v2 (causal mask) + FIXED v4 (node embedding).
# ROOT CAUSE FIX: same as MLP — at 80% sparsity, unobserved nodes all have
# input [0,0,sin,cos]. Without node identity the transformer cannot distinguish
# sensors and collapses to a global prediction. Node embedding added.
class SAITSLite(nn.Module):
    def __init__(self, F, H, n_nodes=325, **k):
        super().__init__()
        self.node_emb = nn.Embedding(n_nodes, H)
        self.proj = nn.Linear(F, H)
        kw = dict(nhead=4, dim_feedforward=H*2, dropout=0.1,
                  activation='gelu', batch_first=True, norm_first=True)
        self.a1 = nn.TransformerEncoderLayer(H, **kw)
        self.a2 = nn.TransformerEncoderLayer(H, **kw)
        self.h1 = nn.Linear(H, 1); self.h2 = nn.Linear(H, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self._n = n_nodes

    @staticmethod
    def _cmask(T, device):
        return torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)

    def forward(self, x):   # x: [N, T, F]
        N, T, _ = x.shape
        cm  = self._cmask(T, x.device)
        ne  = self.node_emb(torch.arange(N, device=x.device))  # [N, H]
        h   = self.proj(x) + ne.unsqueeze(1)                   # [N, T, H]
        h1  = self.a1(h,  src_mask=cm)
        h2  = self.a2(h1, src_mask=cm)
        a   = torch.sigmoid(self.alpha)
        return a * self.h1(h1) + (1-a) * self.h2(h2)

eval_nw(train_nw(SAITSLite, "SAITS", n_nodes=NUM_NODES),
        "11. SAITS-lite (causal Attn + node-emb, fixed)")

# 12. BRITS-lite — forward GRU only (fixed v2). No node-emb needed:
# GRU hidden state h carries node-specific temporal history, so the model
# accumulates a unique fingerprint per node through recurrence.
class BRITSLite(nn.Module):
    def __init__(self,F,H,**k):
        super().__init__(); self.H=H
        self.gf   = nn.GRU(F*2, H, batch_first=True)
        self.impf  = nn.Linear(H, F)
        self.head  = nn.Linear(H, 1)

    def _run(self,x,m,gru,imp):
        N,T,Ff=x.shape
        h=torch.zeros(1,N,self.H,device=x.device); outs=[]
        for t in range(T):
            xh  = imp(h.squeeze(0))
            xc  = m[:,t,:]*x[:,t,:] + (1-m[:,t,:])*xh
            o,h = gru(torch.cat([xc, m[:,t,:]], -1).unsqueeze(1), h)
            outs.append(o)
        return torch.cat(outs, 1)

    def forward(self,x):
        m  = x[:,:,1:2].expand_as(x)
        hf = self._run(x, m, self.gf, self.impf)
        return self.head(hf)
eval_nw(train_nw(BRITSLite,"BRITS"), "12. BRITS-lite (forward GRU only, fixed)")


  TCN trained.
10. TCN (causal dilated, per-node)                    MAE: 1.9895 +/- 0.0200
  SAITS trained.
11. SAITS-lite (causal Attn + node-emb, fixed)        MAE: 3.4992 +/- 0.0290
  BRITS trained.
12. BRITS-lite (forward GRU only, fixed)              MAE: 1.9662 +/- 0.0220


In [ ]:
# Graph baselines: feat [B,N,T,F] + adj A -> [B,N,T]
GE, GH, GLR = 1200, 64, 1e-3

def train_g(net, label):
    # ROOT CAUSE FIX for graph models: unobserved nodes previously had x=0
    # fed into graph diffusion, spreading zeros into neighbouring nodes'
    # representations. Fix: fill unobserved node values with HA prior
    # (z-scored). The mask feature still tells the model which nodes are real.
    opt=torch.optim.Adam(net.parameters(),lr=GLR)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=GE)
    xg,vg=speed_gpu[:TRAIN_END],valid_gpu[:TRAIN_END]; T_tr,N=xg.shape
    for ep in range(1,GE+1):
        net.train()
        t0=np.random.randint(0,T_tr-48)
        xb=xg[t0:t0+48].T.unsqueeze(0); vb=vg[t0:t0+48].T.unsqueeze(0)
        ha=ha_prior[t0:t0+48].T.unsqueeze(0)   # [1,N,48] z-scored HA prior
        idx=torch.arange(t0,t0+48,device=device)
        s=torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,N,-1)
        c=torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,N,-1)
        mb=(torch.rand(1,N,48,device=device)>SPARSITY).float(); me=mb*vb
        # ▶ FIX: unobserved positions filled with HA (not 0) before graph diffusion
        xi = xb*me + ha*(1-me)   # observed: real value; unobserved: HA prior
        feat=torch.stack([xi[0],me[0],s[0],c[0]],dim=-1).unsqueeze(0)
        pred=net(feat,A_t)
        lm=(mb==0)&(vb>0)
        if not lm.any(): continue
        loss=F.smooth_l1_loss(pred[lm],xb[lm])
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(),0.5)
        opt.step(); sch.step()
    print(f"  {label} trained."); return net

def eval_g(net, label):
    net.eval()
    ES,EL=4500,450
    xev=speed_gpu[ES:ES+EL].T.unsqueeze(0); vev=valid_gpu[ES:ES+EL].T.unsqueeze(0)
    hae=ha_prior[ES:ES+EL].T.unsqueeze(0)   # [1,N,EL] HA prior for fill
    idx=torch.arange(ES,ES+EL,device=device)
    s=torch.sin(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    c=torch.cos(2*np.pi*(idx%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st=torch.tensor(node_stds,device=device).view(1,-1,1)
    mn_=torch.tensor(node_means,device=device).view(1,-1,1)
    maes=[]; _pp=[]; _tt=[]
    with torch.no_grad():
        for seed in EVAL_SEEDS:
            m_np = make_eval_mask_np(seed, EL, NUM_NODES)  # [T,N]
            mev=torch.tensor(m_np, device=device).T.unsqueeze(0)  # [1,N,T]
            me=mev*vev
            # ▶ FIX: fill unobserved with HA prior (not 0) before graph diffusion
            xi = xev*me + hae*(1-me)
            feat=torch.stack([xi[0],me[0],s[0],c[0]],dim=-1).unsqueeze(0)
            pred=net(feat,A_t)
            pk=(pred*st+mn_).clamp(CLAMP_LO, CLAMP_HI); tk=(xev*st+mn_).clamp(CLAMP_LO, CLAMP_HI)
            sm=(mev==0)&(vev>0)
            maes.append(torch.abs(pk[sm]-tk[sm]).mean().item())
            _pp.append(pk[sm].cpu().numpy()); _tt.append(tk[sm].cpu().numpy())
    arr=np.array(maes)
    print(f"{label:<52}  MAE: {arr.mean():.4f} +/- {arr.std():.4f}")
    RESULTS[label]=arr
    EXT_PRED[label]=(np.concatenate(_pp), np.concatenate(_tt))

class DiffGCN(nn.Module):
    def __init__(self,i,o,K=2):
        super().__init__(); self.K=K; self.l=nn.Linear((K+1)*i,o)
    def forward(self,x,A):
        out=[x]; Ax=x
        for _ in range(self.K): Ax=torch.matmul(A.unsqueeze(0),Ax); out.append(Ax)
        return self.l(torch.cat(out,-1))

# 13. DCRNN-lite — forward GRU + graph diffusion: strictly causal, no fix needed.
class DCRNNLite(nn.Module):
    def __init__(self,F=4,H=64,**k):
        super().__init__(); self.H=H
        self.gr=DiffGCN(F+H,H); self.gu=DiffGCN(F+H,H); self.gc=DiffGCN(F+H,H)
        self.head=nn.Linear(H,1)
    def _step(self,x,h,A):
        xu=torch.cat([x,h],-1)
        r=torch.sigmoid(self.gr(xu,A)); u=torch.sigmoid(self.gu(xu,A))
        c=torch.tanh(self.gc(torch.cat([x,r*h],-1),A))
        return u*h+(1-u)*c
    def forward(self,feat,A):
        B,N,T,_=feat.shape; h=torch.zeros(B,N,self.H,device=feat.device); outs=[]
        for t in range(T): h=self._step(feat[:,:,t,:],h,A); outs.append(self.head(h))
        return torch.stack(outs,2).squeeze(-1)
dcrnn=DCRNNLite(H=GH).to(device); train_g(dcrnn,"DCRNN-lite"); eval_g(dcrnn,"13. DCRNN-lite (DiffGCN + GRU)")

# 14. ASTGCN-lite — FIXED: added causal mask to temporal self-attention.
# Original used full attention with no mask → position t attended to t+k (future leakage).
# Fix: same upper-triangular -inf causal mask as SAITS fix above.
class ASTGCNLite(nn.Module):
    def __init__(self,F=4,H=64,**k):
        super().__init__()
        self.proj=nn.Linear(F,H); self.gcn=DiffGCN(H,H)
        self.ta=nn.TransformerEncoderLayer(H,4,H*2,dropout=0.1,
            activation="gelu",batch_first=True,norm_first=True)
        self.head=nn.Linear(H,1)

    @staticmethod
    def _cmask(T, device):
        return torch.triu(torch.full((T, T), float('-inf'), device=device), diagonal=1)

    def forward(self,feat,A):
        B,N,T,_=feat.shape; h=self.proj(feat)
        hs=h.permute(0,2,1,3).reshape(B*T,N,-1)
        hs=self.gcn(hs,A).reshape(B,T,N,-1).permute(0,2,1,3)
        h=h+hs; cm=self._cmask(T, feat.device)
        ht=self.ta(h.reshape(B*N,T,-1), src_mask=cm).reshape(B,N,T,-1)  # causal
        return self.head(h+ht).squeeze(-1)
astgcn=ASTGCNLite(H=GH).to(device); train_g(astgcn,"ASTGCN-lite"); eval_g(astgcn,"14. ASTGCN-lite (GCN + Causal Attn, fixed)")

# 15. GWN-lite — causal dilated convolutions: strictly left-to-right, no fix needed.
class GWNLite(nn.Module):
    def __init__(self,F=4,H=64,num_nodes=325,**k):
        super().__init__()
        self.E1=nn.Parameter(torch.randn(num_nodes,10))
        self.E2=nn.Parameter(torch.randn(10,num_nodes))
        self.proj=nn.Linear(F,H)
        self.convs=nn.ModuleList([nn.Conv1d(H,H*2,2,dilation=2**i) for i in range(4)])
        self.gcn=DiffGCN(H,H,K=1); self.head=nn.Linear(H,1)
    def forward(self,feat,A_static):
        B,N,T,_=feat.shape
        Aa=torch.softmax(torch.relu(self.E1@self.E2),-1)
        Am=0.5*(A_static+Aa)
        h=self.proj(feat).permute(0,1,3,2).reshape(B*N,-1,T)
        skip=[]
        for cv in self.convs:
            d=cv.dilation[0]; p=(cv.kernel_size[0]-1)*d
            g=cv(F.pad(h,(p,0))); g1,g2=g.chunk(2,1)
            s=torch.tanh(g1)*torch.sigmoid(g2)
            h=h+s[:,:,:T]; skip.append(h)
        h=sum(skip).reshape(B,N,-1,T).permute(0,3,1,2).reshape(B*T,N,-1)
        h=self.gcn(h,Am).reshape(B,T,N,-1).permute(0,2,1,3)
        return self.head(h).squeeze(-1)
gwn=GWNLite(H=GH,num_nodes=NUM_NODES).to(device); train_g(gwn,"GWN-lite"); eval_g(gwn,"15. GWN-lite (Adaptive GCN + Gated TCN)")


In [ ]:
# V5 hyperparameters + shared multi-seed train/eval helpers
HIDDEN_DIM_V5   = 128
N_LAYERS_V5     = 6
TRAIN_EPOCHS_V5 = 2000
BATCH_SIZE_V5   = 4

# Training (initialization) seeds -> error bars reflect training variance, not
# just eval-mask variance. Eval masks stay shared across all models/seeds.
TRAIN_SEEDS = [0, 1, 2]


def _train_st(ctor, epochs, batch_size, train_seed, label):
    """Train one spatiotemporal model (V5 or MARST share this exact loop).
    Re-seeds torch+numpy so each training seed gives a different init/batch order."""
    torch.manual_seed(train_seed)
    np.random.seed(train_seed)
    net = ctor()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(1, epochs + 1):
        net.train()
        sparsity_ep = min(SPARSITY, 0.60 + (SPARSITY - 0.60) * min(1.0, (ep - 1) / 600))
        t0_list = np.random.randint(0, TRAIN_END - BATCH_TIME, batch_size)
        xs, has, ss, cs, ms, vs = [], [], [], [], [], []
        for t0 in t0_list:
            xs .append(speed_gpu[t0:t0+BATCH_TIME].T)
            has.append(ha_prior[t0:t0+BATCH_TIME].T)
            vs .append(valid_gpu[t0:t0+BATCH_TIME].T)
            ti = torch.arange(t0, t0 + BATCH_TIME, device=device)
            ss.append(torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,-1).expand(NUM_NODES,-1))
            cs.append(torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,-1).expand(NUM_NODES,-1))
            ms.append((torch.rand(NUM_NODES, BATCH_TIME, device=device) > sparsity_ep).float())
        xb, hb, sb, cb, mb, vb = map(torch.stack, (xs, has, ss, cs, ms, vs))
        m_eff = mb * vb
        p = net(xb * m_eff, m_eff, sb, cb, hb)
        lm = (mb == 0) & (vb > 0)
        if not lm.any():
            continue
        loss = F.smooth_l1_loss(p[lm], xb[lm], beta=HUBER_BETA)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
        opt.step(); sch.step()
        if ep % 500 == 0:
            msg = f"  [{label} seed {train_seed}] ep {ep:4d} | loss {loss.item():.4f}"
            if getattr(net, 'last_pi', None) is not None:      # MARST anchor mix
                pi = net.last_pi.mean(dim=0)
                msg += f" | pi[LOCF={pi[0]:.2f} HA={pi[1]:.2f} KNN={pi[2]:.2f}]"
            print(msg, flush=True)
    return net


def _eval_st_mae(net):
    """Per-eval-mask MAE (km/h) over the shared EVAL_SEEDS masks for one trained model."""
    net.eval()
    ES, EL = EVAL_START, EVAL_LEN
    x_ev  = speed_gpu[ES:ES+EL].T.unsqueeze(0)
    ha_ev = ha_prior [ES:ES+EL].T.unsqueeze(0)
    v_ev  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
    ti = torch.arange(ES, ES+EL, device=device)
    ts = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st = torch.tensor(node_stds,  device=device).view(1,-1,1)
    mn = torch.tensor(node_means, device=device).view(1,-1,1)
    out = []
    with torch.no_grad():
        for seed in EVAL_SEEDS:
            m_ev = torch.tensor(make_eval_mask_np(seed, EL, NUM_NODES), device=device).T.unsqueeze(0)
            m_eff = m_ev * v_ev
            preds = []
            for c0 in range(0, EL, BATCH_TIME):
                c1 = min(c0 + BATCH_TIME, EL)
                preds.append(net(x_ev[:,:,c0:c1]*m_eff[:,:,c0:c1], m_eff[:,:,c0:c1],
                                 ts[:,:,c0:c1], tc[:,:,c0:c1], ha_ev[:,:,c0:c1]))
            p  = torch.cat(preds, dim=2)
            pk = (p    * st + mn).clamp(CLAMP_LO, CLAMP_HI)
            tk = (x_ev * st + mn).clamp(CLAMP_LO, CLAMP_HI)
            sm = (m_ev == 0) & (v_ev > 0)
            out.append(torch.abs(pk[sm] - tk[sm]).mean().item())
    return np.array(out)


# --- Train V5 across TRAIN_SEEDS -------------------------------------------
ctor_v5 = lambda: MaskedSTTransformerV5(
    NUM_NODES, A_t, node_means_t, node_stds_t,
    hidden=HIDDEN_DIM_V5, n_heads=N_HEADS, n_layers=N_LAYERS_V5, dropout=DROPOUT).to(device)

_n_params_v5 = sum(p.numel() for p in ctor_v5().parameters() if p.requires_grad)
print(f"Training V5 x{len(TRAIN_SEEDS)} seeds [{DATASET_NAME}] | params: {_n_params_v5/1e6:.2f}M | "
      f"hidden={HIDDEN_DIM_V5} layers={N_LAYERS_V5} epochs={TRAIN_EPOCHS_V5} | base=soft_locf")

v5_mask_mat = []          # [n_train_seeds, n_eval_masks] MAE
for si, tseed in enumerate(TRAIN_SEEDS):
    net = _train_st(ctor_v5, TRAIN_EPOCHS_V5, BATCH_SIZE_V5, tseed, "V5")
    v5_mask_mat.append(_eval_st_mae(net))
    if si == 0:
        net_v5 = net          # representative run for extended metrics / figures
    else:
        del net
        if torch.cuda.is_available(): torch.cuda.empty_cache()
v5_mask_mat = np.array(v5_mask_mat)
print("V5 per-seed MAE (mean over eval masks):", np.round(v5_mask_mat.mean(1), 4))


In [ ]:
# V5 result: mean +/- std over training seeds (each seed averaged over the shared eval masks)
mae_v5_seeds = v5_mask_mat.mean(axis=1)        # [n_train_seeds] -> training-variance error bar
print(f"V5 MAE: {mae_v5_seeds.mean():.4f} +/- {mae_v5_seeds.std():.4f} {VALUE_UNIT}  "
      f"(over {len(TRAIN_SEEDS)} training seeds x {len(EVAL_SEEDS)} eval masks)")
print("  per training seed:", {s: round(float(v), 4) for s, v in zip(TRAIN_SEEDS, mae_v5_seeds)})


## MARST — Multi-Anchor Residual Spatiotemporal Transformer

A new model trained **alongside V5** for direct comparison. MARST keeps V5's
spatiotemporal trunk (alternating temporal/spatial `STBlock`, causal temporal
mask, mask-aware spatial attention), residual head, and meta-gate `α`, but
generalises V5's *single* anchor (`soft_locf`) to a **learned mixture of three
causal anchors** at every (sensor, time) cell:

- **a_LOCF** — last observed value at the sensor (recency)
- **a_HA** — per-sensor historical average for the time-of-day (seasonality)
- **a_KNN** — mean of *currently-observed* graph neighbours (spatial context)

A small head emits a softmax `π ∈ Δ²`; the blended anchor is
`a = π_LOCF·a_LOCF + π_HA·a_HA + π_KNN·a_KNN`, and the prediction is
`ŷ = a + α·r`. Same anti-leak invariants hold: anchors are causal and the
adjacency diagonal is zero, so a blind node never reads its own masked value.
Same hyperparameters as V5 (hidden 128, 6 layers, 2000 epochs, 60%→80%
masking curriculum) for a fair head-to-head.

Both models are trained over **3 initialization seeds** (`TRAIN_SEEDS`); the reported MAE is **mean ± std across training seeds** (each averaged over the 5 shared eval masks), and a **paired t-test** across the shared eval masks reports MARST-vs-V5 significance. The representative first-seed run is reused for the qualitative figures and the extended-metrics table.


In [ ]:
# MARST hyperparameters (matched to V5 for a fair head-to-head).
# Reuses _train_st / _eval_st_mae and TRAIN_SEEDS defined in the V5 cell above.
HIDDEN_DIM_MARST   = 128
N_LAYERS_MARST     = 6
TRAIN_EPOCHS_MARST = 2000
BATCH_SIZE_MARST   = 4

ctor_marst = lambda: MaskedSTTransformerMARST(
    NUM_NODES, A_t, node_means_t, node_stds_t,
    hidden=HIDDEN_DIM_MARST, n_heads=N_HEADS, n_layers=N_LAYERS_MARST, dropout=DROPOUT).to(device)

_n_params_marst = sum(p.numel() for p in ctor_marst().parameters() if p.requires_grad)
print(f"Training MARST x{len(TRAIN_SEEDS)} seeds [{DATASET_NAME}] | params: {_n_params_marst/1e6:.2f}M | "
      f"hidden={HIDDEN_DIM_MARST} layers={N_LAYERS_MARST} epochs={TRAIN_EPOCHS_MARST} | "
      f"anchors=3 (LOCF/HA/KNN) softmax-mixed")

marst_mask_mat = []       # [n_train_seeds, n_eval_masks] MAE
for si, tseed in enumerate(TRAIN_SEEDS):
    net = _train_st(ctor_marst, TRAIN_EPOCHS_MARST, BATCH_SIZE_MARST, tseed, "MARST")
    marst_mask_mat.append(_eval_st_mae(net))
    if si == 0:
        net_marst = net       # representative run for extended metrics / figures
    else:
        del net
        if torch.cuda.is_available(): torch.cuda.empty_cache()
marst_mask_mat = np.array(marst_mask_mat)
print("MARST per-seed MAE (mean over eval masks):", np.round(marst_mask_mat.mean(1), 4))


In [ ]:
# MARST result + MARST-vs-V5 significance test
mae_marst_seeds = marst_mask_mat.mean(axis=1)      # [n_train_seeds]
print(f"MARST MAE: {mae_marst_seeds.mean():.4f} +/- {mae_marst_seeds.std():.4f} {VALUE_UNIT}  "
      f"(over {len(TRAIN_SEEDS)} training seeds x {len(EVAL_SEEDS)} eval masks)")
print("  per training seed:", {s: round(float(v), 4) for s, v in zip(TRAIN_SEEDS, mae_marst_seeds)})

# --- Paired significance test: MARST vs V5 ---------------------------------
# Pair across the shared eval masks (training seeds averaged out): each of the
# EVAL_SEEDS masks hides the SAME positions for both models, so it is a clean
# paired condition. delta > 0 => MARST has lower error.
v5_by_mask    = v5_mask_mat.mean(axis=0)           # [n_eval_masks]
marst_by_mask = marst_mask_mat.mean(axis=0)
delta = v5_by_mask - marst_by_mask
print()
print("="*64)
print("MARST vs V5  (paired across shared eval masks)")
print("="*64)
print(f"mean delta = {delta.mean():+.4f} {VALUE_UNIT} "
      f"({100*delta.mean()/v5_by_mask.mean():+.1f}% of V5 MAE), n = {delta.size} masks")
try:
    from scipy import stats
    t_stat, p_val = stats.ttest_rel(v5_by_mask, marst_by_mask)
    verdict = "SIGNIFICANT" if p_val < 0.05 else "not significant"
    print(f"paired t-test: t = {t_stat:+.3f},  p = {p_val:.4g}  ->  {verdict} at alpha=0.05")
except ImportError:
    n  = delta.size
    sd = delta.std(ddof=1)
    t_stat = delta.mean() / (sd / np.sqrt(n) + 1e-12)
    print(f"paired t-stat = {t_stat:+.3f} (n={n}); install scipy for a p-value")
print(f"headline: V5 {mae_v5_seeds.mean():.4f}+/-{mae_v5_seeds.std():.4f}  |  "
      f"MARST {mae_marst_seeds.mean():.4f}+/-{mae_marst_seeds.std():.4f} {VALUE_UNIT}")
print("Note: 3 training seeds / 5 eval masks. Raise TRAIN_SEEDS for stronger significance claims.")
print("="*64)


In [ ]:
# Final comparison table
RESULTS["18. MaskedSTTransformerV5 (ours)"] = mae_v5_seeds
RESULTS["19. MARST (ours, multi-anchor)"]   = mae_marst_seeds

ha_mae = RESULTS["1. Historical Average (HA)"].mean()
OURS = {"18. MaskedSTTransformerV5 (ours)", "19. MARST (ours, multi-anchor)"}
print()
print("="*72)
print(f"{'PEMS-BAY IMPUTATION BENCHMARK  (80% Sparsity, 5 seeds)':^72}")
print("="*72)
print(f"{'Model':<52} {'MAE':>8}  {'Std':>6}  {'vs HA':>7}")
print("-"*72)
for name, arr in sorted(RESULTS.items(), key=lambda kv: kv[1].mean()):
    tag = " < OURS" if name in OURS else ""
    diff = arr.mean() - ha_mae
    print(f"{name:<52} {arr.mean():>8.4f}  {arr.std():>6.4f}  {diff:>+7.4f}{tag}")
print("-"*72)
for label, key in [("V5", "18. MaskedSTTransformerV5 (ours)"),
                   ("MARST", "19. MARST (ours, multi-anchor)")]:
    d = ha_mae - RESULTS[key].mean()
    print(f"Ours ({label:<5}) vs HA: {d:+.4f} {VALUE_UNIT}  ({100*d/ha_mae:+.1f}%)")
d_mv = RESULTS["18. MaskedSTTransformerV5 (ours)"].mean() - RESULTS["19. MARST (ours, multi-anchor)"].mean()
print(f"MARST vs V5:        {d_mv:+.4f} {VALUE_UNIT}  ({100*d_mv/RESULTS['18. MaskedSTTransformerV5 (ours)'].mean():+.1f}%)")
print(f"std: V5/MARST = training-seed variance (n={len(TRAIN_SEEDS)}); baselines = eval-mask variance (n={len(EVAL_SEEDS)})")
print("="*72)


In [ ]:
# ── Extended Evaluation Metrics — ALL models ────────────────────────────────
# RMSE / MAPE / R² / Pearson / MBE / MedAE / Hit@tol for every model. Baselines'
# predictions were stashed into EXT_PRED during their eval (eval_fn / eval_nw /
# eval_g); here we add the representative V5 and MARST runs, then score everyone
# on the same held-out positions (all EVAL_SEEDS).
import numpy as np, torch, re

def _run_model_ext_full(net):
    """Flattened (pred_kmh, true_kmh) over all EVAL_SEEDS held-out positions."""
    net.eval()
    ES, EL = EVAL_START, EVAL_LEN
    x_ev  = speed_gpu[ES:ES+EL].T.unsqueeze(0)
    ha_ev = ha_prior [ES:ES+EL].T.unsqueeze(0)
    v_ev  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
    ti = torch.arange(ES, ES+EL, device=device)
    ts = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st = torch.tensor(node_stds,  device=device).view(1,-1,1)
    mn = torch.tensor(node_means, device=device).view(1,-1,1)
    pp, tt = [], []
    with torch.no_grad():
        for seed in EVAL_SEEDS:
            m_ev = torch.tensor(make_eval_mask_np(seed, EL, NUM_NODES), device=device).T.unsqueeze(0)
            me   = m_ev * v_ev
            preds = []
            for c0 in range(0, EL, BATCH_TIME):
                c1 = min(c0 + BATCH_TIME, EL)
                preds.append(net(x_ev[:,:,c0:c1]*me[:,:,c0:c1], me[:,:,c0:c1],
                                 ts[:,:,c0:c1], tc[:,:,c0:c1], ha_ev[:,:,c0:c1]))
            p  = torch.cat(preds, dim=2)
            pk = (p    * st + mn).clamp(CLAMP_LO, CLAMP_HI)
            tk = (x_ev * st + mn).clamp(CLAMP_LO, CLAMP_HI)
            sm = (m_ev == 0) & (v_ev > 0)
            pp.append(pk[sm].cpu().numpy()); tt.append(tk[sm].cpu().numpy())
    return np.concatenate(pp), np.concatenate(tt)

# Add the representative V5 / MARST runs (baselines already in EXT_PRED)
EXT_PRED["18. MaskedSTTransformerV5 (ours)"] = _run_model_ext_full(net_v5)
EXT_PRED["19. MARST (ours, multi-anchor)"]   = _run_model_ext_full(net_marst)

def extended_metrics(pred, true):
    err   = pred - true
    mae   = np.abs(err).mean()
    rmse  = np.sqrt((err**2).mean())
    mape  = (np.abs(err) / np.maximum(true, 1.0)).mean() * 100
    ss_r  = (err**2).sum();  ss_t = ((true - true.mean())**2).sum()
    r2    = 1 - ss_r / ss_t
    mbe   = err.mean()
    medae = np.median(np.abs(err))
    pear  = float(np.corrcoef(pred, true)[0, 1])
    hit5  = (np.abs(err) < HIT_TOL[0]).mean() * 100
    hit10 = (np.abs(err) < HIT_TOL[1]).mean() * 100
    return dict(MAE=mae, RMSE=rmse, MedAE=medae, MAPE=mape,
                R2=r2, Pearson=pear, MBE=mbe, Hit5=hit5, Hit10=hit10)

def _lead_num(k):
    m = re.match(r'\s*(\d+)', k); return int(m.group(1)) if m else 999

# Score every model, ordered by its benchmark number
EXT_METRICS = {k: extended_metrics(*EXT_PRED[k]) for k in sorted(EXT_PRED, key=_lead_num)}
_best_name  = min(EXT_METRICS, key=lambda k: EXT_METRICS[k]["MAE"])

cols_ext = ["MAE", "RMSE", "MedAE", "MAPE%", "R2", "Pearson", "MBE",
            f"Hit@{HIT_TOL[0]:g}", f"Hit@{HIT_TOL[1]:g}"]
mw = 34; w = 11
W = mw + w * len(cols_ext)
print("=" * W)
print(f"{'EXTENDED EVALUATION METRICS  --  ' + DATASET_NAME + '  (80% sparsity, ' + str(len(EVAL_SEEDS)) + ' eval masks)':^{W}}")
print("=" * W)
print(f"{'Model':<{mw}}" + "".join(f"{c:>{w}}" for c in cols_ext))
print("-" * W)
for name, m in EXT_METRICS.items():
    label = name.split(". ", 1)[-1][:mw-2]
    row = (f"{label:<{mw}}"
           f"{m['MAE']:>{w}.4f}{m['RMSE']:>{w}.4f}{m['MedAE']:>{w}.4f}"
           f"{m['MAPE']:>{w}.2f}{m['R2']:>{w}.4f}{m['Pearson']:>{w}.4f}"
           f"{m['MBE']:>{w}.4f}{m['Hit5']:>{w}.2f}{m['Hit10']:>{w}.2f}")
    print(row + ("  <-- BEST MAE" if name == _best_name else ""))
print("=" * W)


## Ablation Study — V5 component leave-one-out

Each variant disables **one** of V5's additions; everything else is identical to the full model (same training budget, seeds, masking curriculum). `delta MAE > 0` means removing that component *hurt* — i.e. it was contributing. Runs on the dataset selected in the config cell.

Anti-leak invariants are preserved in every variant: masked positions are zeroed, adjacency has no self-loops, LOCF/staleness/soft_locf stay causal, and stats are train-only.

In [ ]:
# V5 component leave-one-out ablation
ABLATION_EPOCHS = 800   # trimmed from full V5 (2000) to save time; ablation is a relative comparison
ABLATION_BATCH  = BATCH_SIZE_V5
ABLATION_SEEDS  = EVAL_SEEDS

ABLATIONS = {
    'Full V5 (all features)':   dict(),
    '- 2-hop neighbour mean':   dict(use_2hop=False),
    '- soft_locf (base=locf)':  dict(use_soft_locf=False, base='locf'),
    '- staleness':              dict(use_staleness=False),
    '- 1-hop neighbour mean':   dict(use_n_mean=False),
    '- node embedding':         dict(use_node_emb=False),
}

def _train_ablation(flags, epochs):
    net = MaskedSTTransformerV5Ablate(
        NUM_NODES, A_t, node_means_t, node_stds_t,
        hidden=HIDDEN_DIM_V5, n_heads=N_HEADS, n_layers=N_LAYERS_V5,
        dropout=DROPOUT, **flags).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(1, epochs + 1):
        net.train()
        sparsity_ep = min(SPARSITY, 0.60 + (SPARSITY - 0.60) * min(1.0, (ep - 1) / 600))
        t0_list = np.random.randint(0, TRAIN_END - BATCH_TIME, ABLATION_BATCH)
        xl, hl, sl, cl, ml, vl = [], [], [], [], [], []
        for t0 in t0_list:
            xl.append(speed_gpu[t0:t0+BATCH_TIME].T)
            hl.append(ha_prior[t0:t0+BATCH_TIME].T)
            vl.append(valid_gpu[t0:t0+BATCH_TIME].T)
            ti = torch.arange(t0, t0+BATCH_TIME, device=device)
            sl.append(torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,-1).expand(NUM_NODES,-1))
            cl.append(torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,-1).expand(NUM_NODES,-1))
            ml.append((torch.rand(NUM_NODES, BATCH_TIME, device=device) > sparsity_ep).float())
        xb, hb, sb, cb, mb, vb = map(torch.stack, (xl, hl, sl, cl, ml, vl))
        me = mb * vb
        p = net(xb * me, me, sb, cb, hb)
        lm = (mb == 0) & (vb > 0)
        if not lm.any():
            continue
        loss = F.smooth_l1_loss(p[lm], xb[lm], beta=HUBER_BETA)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
        opt.step(); sch.step()
    return net

def _eval_ablation(net):
    net.eval()
    x_ev  = speed_gpu[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    ha_ev = ha_prior [EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    v_ev  = valid_gpu[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    ti = torch.arange(EVAL_START, EVAL_START+EVAL_LEN, device=device)
    ts = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st = torch.tensor(node_stds,  device=device).view(1,-1,1)
    mn = torch.tensor(node_means, device=device).view(1,-1,1)
    maes = []
    with torch.no_grad():
        for seed in ABLATION_SEEDS:
            m_ev = torch.tensor(make_eval_mask_np(seed, EVAL_LEN, NUM_NODES), device=device).T.unsqueeze(0)
            me = m_ev * v_ev
            preds = []
            for c0 in range(0, EVAL_LEN, BATCH_TIME):
                c1 = min(c0 + BATCH_TIME, EVAL_LEN)
                preds.append(net(x_ev[:,:,c0:c1]*me[:,:,c0:c1], me[:,:,c0:c1],
                                 ts[:,:,c0:c1], tc[:,:,c0:c1], ha_ev[:,:,c0:c1]))
            p  = torch.cat(preds, dim=2)
            pk = (p * st + mn).clamp(CLAMP_LO, CLAMP_HI)
            tk = (x_ev * st + mn).clamp(CLAMP_LO, CLAMP_HI)
            sm = (m_ev == 0) & (v_ev > 0)
            e = torch.abs(pk[sm] - tk[sm])
            e = e[torch.isfinite(e)]  # drop rare all-blind-timestep NaNs (extreme sparsity)
            maes.append(e.mean().item())
    return np.array(maes)

ABLATION_RESULTS = {}
for name, flags in ABLATIONS.items():
    print(f"Training ablation: {name} ...", flush=True)
    net_ab = _train_ablation(flags, ABLATION_EPOCHS)
    ABLATION_RESULTS[name] = _eval_ablation(net_ab)
    del net_ab
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"  {name:<28} MAE: {ABLATION_RESULTS[name].mean():.4f} +/- {ABLATION_RESULTS[name].std():.4f}")

full = ABLATION_RESULTS['Full V5 (all features)'].mean()
title = f"V5 ABLATION  ({DATASET_NAME}, {int(SPARSITY*100)}% sparsity, {len(ABLATION_SEEDS)} seeds)"
print("\n" + "=" * 66)
print(f"{title:^66}")
print("=" * 66)
print(f"{'Variant':<30}{'MAE':>9}{'Std':>9}{'dMAE':>9}{'%':>9}")
print("-" * 66)
for name, arr in ABLATION_RESULTS.items():
    d = arr.mean() - full
    pct = 100 * d / full
    print(f"{name:<30}{arr.mean():>9.4f}{arr.std():>9.4f}{d:>+9.4f}{pct:>+8.1f}%")
print("=" * 66)
print("dMAE > 0 (positive) => removing the component HURTS, i.e. it was helping the full model.")

ABLATION_SAVE = {k: [float(x) for x in v] for k, v in ABLATION_RESULTS.items()}


## Sparsity Sensitivity

Evaluate the **trained** V5 (against LOCF and Historical-Average references) as the fraction of blind sensors grows. Training sparsity was fixed at 80%; this probes robustness/extrapolation. Same anti-leak masking: only `(masked) & (valid)` positions are scored, blind sensors are zeroed.

In [ ]:
# Sparsity-sensitivity sweep on the already-trained V5
SPARSITY_LEVELS = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

def _eval_at_sparsity(model, sparsity, kind='model'):
    x_ev  = speed_gpu[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    ha_ev = ha_prior [EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    v_ev  = valid_gpu[EVAL_START:EVAL_START+EVAL_LEN].T.unsqueeze(0)
    ti = torch.arange(EVAL_START, EVAL_START+EVAL_LEN, device=device)
    ts = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    st = torch.tensor(node_stds,  device=device).view(1,-1,1)
    mn = torch.tensor(node_means, device=device).view(1,-1,1)
    maes = []
    with torch.no_grad():
        for seed in EVAL_SEEDS:
            m_ev = torch.tensor(make_eval_mask_np(seed, EVAL_LEN, NUM_NODES, sparsity=sparsity),
                                device=device).T.unsqueeze(0)
            me = m_ev * v_ev
            if kind == 'model':
                preds = []
                for c0 in range(0, EVAL_LEN, BATCH_TIME):
                    c1 = min(c0 + BATCH_TIME, EVAL_LEN)
                    preds.append(model(x_ev[:,:,c0:c1]*me[:,:,c0:c1], me[:,:,c0:c1],
                                       ts[:,:,c0:c1], tc[:,:,c0:c1], ha_ev[:,:,c0:c1]))
                p = torch.cat(preds, dim=2)
            elif kind == 'ha':
                p = ha_ev
            elif kind == 'locf':
                prev = ha_ev[:, :, 0:1].clone(); out = []
                for t in range(EVAL_LEN):
                    obs = x_ev[:, :, t:t+1] * me[:, :, t:t+1]
                    prev = torch.where(me[:, :, t:t+1] > 0, obs, prev)
                    out.append(prev.clone())
                p = torch.cat(out, dim=2)
            pk = (p * st + mn).clamp(CLAMP_LO, CLAMP_HI)
            tk = (x_ev * st + mn).clamp(CLAMP_LO, CLAMP_HI)
            sm = (m_ev == 0) & (v_ev > 0)
            e = torch.abs(pk[sm] - tk[sm])
            e = e[torch.isfinite(e)]  # drop rare all-blind-timestep NaNs (extreme sparsity)
            maes.append(e.mean().item())
    return np.array(maes)

net_v5.eval()
SPARSITY_SWEEP = {'V5 (ours)': {}, 'LOCF': {}, 'Hist. Average': {}}
for sp in SPARSITY_LEVELS:
    SPARSITY_SWEEP['V5 (ours)'][sp]     = _eval_at_sparsity(net_v5, sp, 'model')
    SPARSITY_SWEEP['LOCF'][sp]          = _eval_at_sparsity(None,   sp, 'locf')
    SPARSITY_SWEEP['Hist. Average'][sp] = _eval_at_sparsity(None,   sp, 'ha')
    print(f"sparsity {int(sp*100):2d}%  V5={SPARSITY_SWEEP['V5 (ours)'][sp].mean():.4f}  "
          f"LOCF={SPARSITY_SWEEP['LOCF'][sp].mean():.4f}  "
          f"HA={SPARSITY_SWEEP['Hist. Average'][sp].mean():.4f}")

title = f"SPARSITY SENSITIVITY  ({DATASET_NAME})"
keys = list(SPARSITY_SWEEP)
W = 11 + 16 * len(keys)
print("\n" + "=" * W)
print(f"{title:^{W}}")
print("=" * W)
print(f"{'Sparsity':>10}" + "".join(f"{k:>16}" for k in keys))
print("-" * W)
for sp in SPARSITY_LEVELS:
    print(f"{int(sp*100):>9}%" + "".join(f"{SPARSITY_SWEEP[k][sp].mean():>16.4f}" for k in keys))
print("=" * W)

SPARSITY_SAVE = {k: {str(sp): [float(x) for x in v] for sp, v in d.items()}
                 for k, d in SPARSITY_SWEEP.items()}


## Save run + Cross-Dataset Aggregation

The first cell writes `results_<DATASET>.json` (MAE table, extended metrics, ablation, sparsity sweep). Run the notebook once per dataset, then the aggregation cell stitches every saved run into a single cross-dataset table.

In [ ]:
# Save this run's results for cross-dataset aggregation
def _arr2list(d):
    return {k: [float(x) for x in v] for k, v in d.items()}

save_obj = dict(
    dataset=DATASET_NAME,
    kind=CFG['kind'],
    sparsity=SPARSITY,
    num_nodes=int(NUM_NODES),
    seeds=EVAL_SEEDS,
    hit_tol=[float(HIT_TOL[0]), float(HIT_TOL[1])],
    regime_edges=[float(REGIME_EDGES[0]), float(REGIME_EDGES[1])],
    mae=_arr2list(RESULTS),
    ext_metrics=({k: {m: float(val) for m, val in d.items()} for k, d in EXT_METRICS.items()}
                 if 'EXT_METRICS' in globals() else {}),
    ablation=globals().get('ABLATION_SAVE', {}),
    sparsity_sweep=globals().get('SPARSITY_SAVE', {}),
)
out_path = f"results_{DATASET_NAME}.json"
with open(out_path, 'w') as f:
    json.dump(save_obj, f, indent=2)
print(f"Saved {out_path}")
print(f"  models in MAE table : {len(save_obj['mae'])}")
print(f"  extended-metric rows: {len(save_obj['ext_metrics'])}")
print(f"  ablation variants   : {len(save_obj['ablation'])}")
print(f"  sparsity levels     : {len(save_obj['sparsity_sweep'].get('V5 (ours)', {}))}")


In [ ]:
# Cross-dataset aggregation: loads every results_<DATASET>.json present.
import glob
files = sorted(glob.glob('results_*.json'))
runs = {}
for fp in files:
    try:
        o = json.load(open(fp))
        runs[o['dataset']] = o
    except Exception as e:
        print(f"skip {fp}: {e}")
print(f"Found {len(runs)} dataset run(s): {list(runs.keys())}\n")

def _mean(o, name):
    v = o['mae'].get(name)
    return float(np.mean(v)) if v else float('nan')

if runs:
    ds_names = list(runs.keys())
    ref = ds_names[0]
    model_names = set()
    for o in runs.values():
        model_names |= set(o['mae'].keys())
    model_names = sorted(model_names,
                         key=lambda n: _mean(runs[ref], n) if not np.isnan(_mean(runs[ref], n)) else 1e9)

    W = 46 + 13 * len(ds_names)
    title = f"CROSS-DATASET MAE  ({int(SPARSITY*100)}% sparsity)"
    print("=" * W)
    print(f"{title:^{W}}")
    print("=" * W)
    print(f"{'Model':<46}" + "".join(f"{d:>13}" for d in ds_names))
    print("-" * W)
    for name in model_names:
        row = f"{name:<46}"
        for d in ds_names:
            mv = _mean(runs[d], name)
            row += (f"{mv:>13.4f}" if not np.isnan(mv) else f"{'-':>13}")
        print(row)
    print("=" * W)

    # Extended metrics, per dataset (compact)
    for d, o in runs.items():
        em = o.get('ext_metrics', {})
        if not em:
            continue
        mkeys = ["MAE", "RMSE", "MedAE", "MAPE", "R2", "Pearson", "MBE", "Hit5", "Hit10"]
        tol = o.get("hit_tol", [5, 10])
        hdr = ["MAE", "RMSE", "MedAE", "MAPE", "R2", "Pearson", "MBE",
               f"Hit@{tol[0]:g}", f"Hit@{tol[1]:g}"]
        ww = 10
        WW = 20 + ww * len(mkeys)
        print("\n" + "-" * WW)
        print(f"Extended metrics -- {d}")
        print("-" * WW)
        print(f"{'Model':<20}" + "".join(f"{c:>{ww}}" for c in hdr))
        for mname, mv in em.items():
            print(f"{mname:<20}" + "".join(f"{mv.get(c, float('nan')):>{ww}.3f}" for c in mkeys))
else:
    print("No results_*.json found yet. Run the notebook once per DATASET, then re-run this cell.")


## Visualisations

In [ ]:
# ============================================================================
#  Publication figures (all-model comparison).  PNG @ 300 dpi.
#  Consumes: RESULTS (per-seed MAE), EXT_PRED (flat preds), EXT_METRICS,
#            v5_mask_mat / marst_mask_mat, net_marst.
# ============================================================================
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np, re

plt.rcParams.update({
    'figure.dpi'      : 120,
    'savefig.dpi'     : 300,
    'font.size'       : 12,
    'axes.titlesize'  : 13,
    'axes.labelsize'  : 12,
    'axes.titleweight': 'bold',
    'legend.fontsize' : 9,
    'axes.spines.top' : False,
    'axes.spines.right': False,
    'figure.autolayout': False,
})

# ── Model metadata ─────────────────────────────────────────────────────────
FAMILY_COLOR = {'stat':'#7F7F7F', 'linear':'#9E9E9E', 'nonparam':'#BCBD22',
                'nn':'#1F77B4', 'rnn':'#17BECF', 'conv':'#2CA02C',
                'attn':'#9467BD', 'graph':'#FF7F0E', 'ours':'#D62728'}
FAMILY_LABEL = {'stat':'Statistical', 'linear':'Linear', 'nonparam':'Non-parametric',
                'nn':'Neural', 'rnn':'RNN', 'conv':'TCN', 'attn':'Attention',
                'graph':'Graph', 'ours':'Ours'}
MODEL_FAMILY = {
    '1. Historical Average (HA)':'stat', '2. LOCF (Last-Obs Carried Forward)':'stat',
    '3. Global Mean (per-node train mean)':'stat', '4. Node-wise Ridge Regression':'linear',
    '5. KNN Imputer (k=5, masked-dist)':'nonparam', '6.  MLP (per-node + node-emb)':'nn',
    '7.  LSTM (per-node)':'rnn', '8.  BiLSTM->2L-LSTM (causal, fixed)':'rnn',
    '9.  GRU (per-node)':'rnn', '10. TCN (causal dilated, per-node)':'conv',
    '11. SAITS-lite (causal Attn + node-emb, fixed)':'attn',
    '12. BRITS-lite (forward GRU only, fixed)':'rnn',
    '13. DCRNN-lite (DiffGCN + GRU)':'graph',
    '14. ASTGCN-lite (GCN + Causal Attn, fixed)':'graph',
    '15. GWN-lite (Adaptive GCN + Gated TCN)':'graph',
    '18. MaskedSTTransformerV5 (ours)':'ours', '19. MARST (ours, multi-anchor)':'ours'}
MODEL_SHORT = {
    '1. Historical Average (HA)':'HA', '2. LOCF (Last-Obs Carried Forward)':'LOCF',
    '3. Global Mean (per-node train mean)':'GlobalMean', '4. Node-wise Ridge Regression':'Ridge',
    '5. KNN Imputer (k=5, masked-dist)':'KNN', '6.  MLP (per-node + node-emb)':'MLP',
    '7.  LSTM (per-node)':'LSTM', '8.  BiLSTM->2L-LSTM (causal, fixed)':'BiLSTM',
    '9.  GRU (per-node)':'GRU', '10. TCN (causal dilated, per-node)':'TCN',
    '11. SAITS-lite (causal Attn + node-emb, fixed)':'SAITS-lite',
    '12. BRITS-lite (forward GRU only, fixed)':'BRITS-lite',
    '13. DCRNN-lite (DiffGCN + GRU)':'DCRNN-lite',
    '14. ASTGCN-lite (GCN + Causal Attn, fixed)':'ASTGCN-lite',
    '15. GWN-lite (Adaptive GCN + Gated TCN)':'GWN-lite',
    '18. MaskedSTTransformerV5 (ours)':'V5', '19. MARST (ours, multi-anchor)':'MARST'}

V5_KEY, MARST_KEY = '18. MaskedSTTransformerV5 (ours)', '19. MARST (ours, multi-anchor)'
OURS_KEYS = {V5_KEY, MARST_KEY}
def short(k):  return MODEL_SHORT.get(k, k.split('. ', 1)[-1][:12])
def fam(k):    return MODEL_FAMILY.get(k, 'stat')
def mcolor(k):                       # ours stand out; V5 vs MARST distinct
    if k == MARST_KEY: return '#D62728'
    if k == V5_KEY:    return '#2CA02C'
    return FAMILY_COLOR[fam(k)]
def _lead(k):
    m = re.match(r'\s*(\d+)', k); return int(m.group(1)) if m else 999

ha_mae   = RESULTS['1. Historical Average (HA)'].mean()
locf_mae = RESULTS['2. LOCF (Last-Obs Carried Forward)'].mean()
DSET = DATASET_NAME

# ════════════════════════════════════════════════════════════════════════
# Fig 1 — Benchmark: MAE ± std, all models, family-coloured, ours highlighted
# ════════════════════════════════════════════════════════════════════════
items  = sorted(RESULTS.items(), key=lambda kv: kv[1].mean(), reverse=True)
labels = [short(k) for k, _ in items]
maes   = np.array([v.mean() for _, v in items])
stds   = np.array([v.std()  for _, v in items])
colors = [mcolor(k) for k, _ in items]

fig, ax = plt.subplots(figsize=(9, 7.5))
y = np.arange(len(labels))
bars = ax.barh(y, maes, xerr=stds, color=colors, height=0.68,
               error_kw=dict(ecolor='#333', capsize=2.5, elinewidth=1.0))
for k, b in zip([k for k, _ in items], bars):     # outline ours
    if k in OURS_KEYS:
        b.set_edgecolor('black'); b.set_linewidth(1.6)
ax.axvline(ha_mae,   color='#444', ls='--', lw=1.4, alpha=.8, label=f'HA ({ha_mae:.2f})')
ax.axvline(locf_mae, color='#888', ls=':',  lw=1.4, alpha=.8, label=f'LOCF ({locf_mae:.2f})')
ax.set_yticks(y); ax.set_yticklabels(labels)
ax.set_xlabel(f'MAE ({VALUE_UNIT})  —  lower is better')
ax.set_title(f'{DSET} Imputation Benchmark  |  80% Sparsity')
ax.invert_yaxis(); ax.grid(axis='x', alpha=.25); ax.set_xlim(0, maes.max()*1.15)
fams_seen = dict.fromkeys(fam(k) for k, _ in items)
fam_handles = [mpatches.Patch(color=FAMILY_COLOR[f], label=FAMILY_LABEL[f]) for f in fams_seen]
leg1 = ax.legend(handles=fam_handles, loc='lower right', title='Family', framealpha=.95)
ax.add_artist(leg1); ax.legend(loc='upper right', framealpha=.95)
if MARST_KEY in RESULTS:
    bm = RESULTS[MARST_KEY].mean(); bi = labels.index(short(MARST_KEY))
    ax.annotate(f'MARST: +{100*(ha_mae-bm)/ha_mae:.1f}% vs HA', xy=(bm, bi),
                xytext=(bm + .10, bi - 0.9), color='#D62728', fontweight='bold', fontsize=10,
                arrowprops=dict(arrowstyle='->', color='#D62728', lw=1.4))
plt.tight_layout(); plt.savefig('fig1_benchmark.png', bbox_inches='tight'); plt.show()
print('Saved fig1_benchmark.png')

# ════════════════════════════════════════════════════════════════════════
# Fig 2 — Extended-metrics heatmap, all models × 9 metrics (green = best/col)
# ════════════════════════════════════════════════════════════════════════
names  = list(EXT_METRICS.keys())
mkeys  = ["MAE", "RMSE", "MedAE", "MAPE", "R2", "Pearson", "MBE", "Hit5", "Hit10"]
mlabs  = [f"MAE\n({VALUE_UNIT})", f"RMSE\n({VALUE_UNIT})", f"MedAE\n({VALUE_UNIT})", "MAPE\n(%)",
          "R²", "Pearson\nr", f"MBE\n({VALUE_UNIT})", f"Hit\n@{HIT_TOL[0]:g}", f"Hit\n@{HIT_TOL[1]:g}"]
raw = np.array([[EXT_METRICS[n][k] for k in mkeys] for n in names])
norm = np.zeros_like(raw); higher = {"R2", "Pearson", "Hit5", "Hit10"}
for j, k in enumerate(mkeys):
    col = raw[:, j]; rng = col.max() - col.min() + 1e-9
    norm[:, j] = (col-col.min())/rng if k in higher else 1-(col-col.min())/rng
fig, ax = plt.subplots(figsize=(13, 0.5*len(names) + 2))
im = ax.imshow(norm, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(mkeys))); ax.set_xticklabels(mlabs, fontsize=10)
ax.set_yticks(range(len(names)));  ax.set_yticklabels([short(n) for n in names], fontsize=10)
for n_i, n in enumerate(names):
    if n in OURS_KEYS: ax.get_yticklabels()[n_i].set_fontweight('bold')
    for j, k in enumerate(mkeys):
        v = raw[n_i, j]
        txt = f"{v:.3f}" if k in ("R2", "Pearson") else (f"{v:.1f}" if k in ("MAPE","Hit5","Hit10") else f"{v:.3f}")
        ax.text(j, n_i, txt, ha='center', va='center', fontsize=7.5,
                color='black' if .25 < norm[n_i, j] < .8 else 'white')
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.01, label='normalised (1 = best in column)')
ax.set_title(f'Extended Metrics — {DSET}  (green = best per column)')
plt.tight_layout(); plt.savefig('fig2_metrics_heatmap.png', bbox_inches='tight'); plt.show()
print('Saved fig2_metrics_heatmap.png')

# ════════════════════════════════════════════════════════════════════════
# Fig 3 — Error CDF + Hit-rate curves, all models (family-coloured, ours bold)
# ════════════════════════════════════════════════════════════════════════
abs_err = {k: np.abs(p - t) for k, (p, t) in EXT_PRED.items()}
order   = sorted(abs_err, key=_lead)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

ax = axes[0]
for k in order:
    e = abs_err[k]; xs = np.sort(e); ys = np.arange(1, len(xs)+1)/len(xs)
    step = max(1, len(xs)//2000)
    lw, a, z = (2.6, 1.0, 5) if k in OURS_KEYS else (1.0, .55, 2)
    ax.plot(xs[::step], ys[::step], color=mcolor(k), lw=lw, alpha=a, zorder=z)
ax.set_xlim(0, 12); ax.set_ylim(0, 1)
ax.set_xlabel(f'Absolute error ({VALUE_UNIT})'); ax.set_ylabel('Fraction of held-out points')
ax.set_title('Error CDF (higher-left = better)'); ax.grid(alpha=.25)

ax = axes[1]
thr = np.linspace(0, HIT_EPS_MAX, 200)
for k in order:
    e = abs_err[k]
    lw, a, z = (2.6, 1.0, 5) if k in OURS_KEYS else (1.0, .55, 2)
    ax.plot(thr, [(e < x).mean()*100 for x in thr], color=mcolor(k), lw=lw, alpha=a, zorder=z)
for vx in HIT_TOL:
    ax.axvline(vx, color='gray', ls=':', lw=1, alpha=.6)
ax.set_xlim(0, HIT_EPS_MAX); ax.set_ylim(0, 100)
ax.set_xlabel(f'Tolerance ε ({VALUE_UNIT})'); ax.set_ylabel('Hit rate (% within ε)')
ax.set_title('Hit-Rate Curves'); ax.grid(alpha=.25)

fam_handles = [mpatches.Patch(color=FAMILY_COLOR[f], label=FAMILY_LABEL[f])
               for f in dict.fromkeys(fam(k) for k in order)]
fam_handles += [Line2D([], [], color=mcolor(V5_KEY), lw=2.6, label='V5 (ours)'),
                Line2D([], [], color=mcolor(MARST_KEY), lw=2.6, label='MARST (ours)')]
axes[1].legend(handles=fam_handles, loc='lower right', framealpha=.95, ncol=1)
plt.suptitle(f'Cumulative Accuracy — {DSET}  (all models)', fontweight='bold')
plt.tight_layout(); plt.savefig('fig3_cdf_hitrate.png', bbox_inches='tight'); plt.show()
print('Saved fig3_cdf_hitrate.png')

# ════════════════════════════════════════════════════════════════════════
# Fig 4 — RMSE vs MAE scatter (outlier sensitivity), all models
# ════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 7))
for k in sorted(EXT_METRICS, key=_lead):
    m = EXT_METRICS[k]; isours = k in OURS_KEYS
    ax.scatter(m['MAE'], m['RMSE'], s=240 if isours else 90, color=mcolor(k),
               marker='*' if isours else 'o', edgecolors='black' if isours else 'none',
               linewidths=1.2 if isours else 0, zorder=5 if isours else 3, alpha=.9)
    ax.annotate(short(k), (m['MAE'], m['RMSE']), fontsize=8,
                xytext=(4, 3), textcoords='offset points',
                fontweight='bold' if isours else 'normal')
lo = min(m['MAE'] for m in EXT_METRICS.values()) - .1
hi = max(m['MAE'] for m in EXT_METRICS.values()) + .1
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, alpha=.4, label='RMSE = MAE')
ax.set_xlabel(f'MAE ({VALUE_UNIT})'); ax.set_ylabel(f'RMSE ({VALUE_UNIT})')
ax.set_title(f'RMSE vs MAE — {DSET}  (lower-left = better)')
ax.legend(loc='upper left'); ax.grid(alpha=.25)
plt.tight_layout(); plt.savefig('fig4_rmse_vs_mae.png', bbox_inches='tight'); plt.show()
print('Saved fig4_rmse_vs_mae.png')

# ════════════════════════════════════════════════════════════════════════
# Fig 5 — Per-seed stability, all models (strip + mean)
# ════════════════════════════════════════════════════════════════════════
st_items = sorted(RESULTS.items(), key=lambda kv: kv[1].mean())
fig, ax = plt.subplots(figsize=(13, 5))
rng = np.random.default_rng(0)
for i, (k, arr) in enumerate(st_items):
    jit = rng.uniform(-.16, .16, len(arr))
    ax.scatter(np.full(len(arr), i)+jit, arr, s=42, color=mcolor(k),
               alpha=.85, zorder=4, edgecolors='black' if k in OURS_KEYS else 'none', linewidths=.8)
    ax.hlines(arr.mean(), i-.32, i+.32, color=mcolor(k), lw=2.6, zorder=5)
ax.set_xticks(range(len(st_items)))
ax.set_xticklabels([short(k) for k, _ in st_items], rotation=40, ha='right')
for i, (k, _) in enumerate(st_items):
    if k in OURS_KEYS: ax.get_xticklabels()[i].set_fontweight('bold')
ax.set_ylabel(f'MAE ({VALUE_UNIT})'); ax.set_title(f'Per-Seed Stability — {DSET}')
ax.grid(axis='y', alpha=.25)
ax.text(0.005, -0.32, 'Bars = mean. Ours (V5/MARST): 3 training seeds; baselines: 5 eval-mask seeds.',
        transform=ax.transAxes, fontsize=8.5, style='italic', color='#555')
plt.tight_layout(); plt.savefig('fig5_per_seed_stability.png', bbox_inches='tight'); plt.show()
print('Saved fig5_per_seed_stability.png')

# ════════════════════════════════════════════════════════════════════════
# Fig 6 — MARST vs V5 head-to-head: paired-by-mask + learned anchor mixture
# ════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: paired MAE across the shared eval masks (training seeds averaged)
v5_bm    = v5_mask_mat.mean(axis=0)        # [n_eval_masks]
marst_bm = marst_mask_mat.mean(axis=0)
ax = axes[0]
xs = np.arange(len(EVAL_SEEDS))
ax.plot(xs, v5_bm,    'o-', color='#2CA02C', lw=2, ms=8, label='V5')
ax.plot(xs, marst_bm, 's-', color='#D62728', lw=2, ms=8, label='MARST')
for x, a, b in zip(xs, v5_bm, marst_bm):
    ax.plot([x, x], [a, b], color='#aaa', lw=1, zorder=0)
ax.set_xticks(xs); ax.set_xticklabels([f'mask {s}' for s in EVAL_SEEDS], rotation=20)
ax.set_ylabel(f'MAE ({VALUE_UNIT})'); ax.set_title('V5 vs MARST (paired by eval mask)')
ax.legend(); ax.grid(axis='y', alpha=.25)
delta = v5_bm - marst_bm
try:
    from scipy import stats
    _t, _p = stats.ttest_rel(v5_bm, marst_bm)
    sig = f"Δ={delta.mean():+.4f} {VALUE_UNIT}\npaired t-test p={_p:.3g}"
    sig += "  *" if _p < 0.05 else "  (n.s.)"
except Exception:
    sig = f"Δ={delta.mean():+.4f} {VALUE_UNIT} (n={delta.size})"
ax.text(0.03, 0.04, sig, transform=ax.transAxes, fontsize=10, va='bottom',
        bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='#999', alpha=.95))

# Panel B: MARST learned anchor mixture π (mean over eval window)
ax = axes[1]
pi_mean = None
if getattr(net_marst, 'last_pi', None) is not None or hasattr(net_marst, 'anchor_gate'):
    net_marst.eval()
    ES, EL = EVAL_START, EVAL_LEN
    xb_  = speed_gpu[ES:ES+EL].T.unsqueeze(0); hab_ = ha_prior[ES:ES+EL].T.unsqueeze(0)
    vb_  = valid_gpu[ES:ES+EL].T.unsqueeze(0)
    ti = torch.arange(ES, ES+EL, device=device)
    ts = torch.sin(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    tc = torch.cos(2*np.pi*(ti%STEPS_PER_DAY)/STEPS_PER_DAY).view(1,1,-1).expand(1,NUM_NODES,-1)
    m_ = torch.tensor(make_eval_mask_np(EVAL_SEEDS[0], EL, NUM_NODES), device=device).T.unsqueeze(0)
    me_ = m_ * vb_; pis = []
    with torch.no_grad():
        for c0 in range(0, EL, BATCH_TIME):
            c1 = min(c0+BATCH_TIME, EL)
            _ = net_marst(xb_[:,:,c0:c1]*me_[:,:,c0:c1], me_[:,:,c0:c1],
                          ts[:,:,c0:c1], tc[:,:,c0:c1], hab_[:,:,c0:c1])
            pis.append(net_marst.last_pi.mean(0).cpu().numpy())
    pi_mean = np.mean(pis, axis=0)
if pi_mean is not None:
    anames = ['LOCF\n(recency)', 'HA\n(seasonality)', 'KNN\n(spatial)']
    acols  = ['#1f77b4', '#ff7f0e', '#9467bd']
    bars = ax.bar(anames, pi_mean, color=acols, alpha=.9, edgecolor='black', lw=.8)
    for b, v in zip(bars, pi_mean):
        ax.text(b.get_x()+b.get_width()/2, v+.01, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(pi_mean)*1.25); ax.set_ylabel('mean anchor weight  π')
    ax.set_title('MARST learned anchor mixture')
else:
    ax.text(.5, .5, 'anchor mixture unavailable', ha='center', va='center', transform=ax.transAxes)
    ax.set_axis_off()
plt.suptitle(f'Our models — {DSET}: head-to-head & interpretability', fontweight='bold')
plt.tight_layout(); plt.savefig('fig6_marst_vs_v5.png', bbox_inches='tight'); plt.show()
print('Saved fig6_marst_vs_v5.png')

print('\nAll publication figures saved: fig1..fig6 (PNG @ 300 dpi)')


## Ablation & Sparsity Plots

Saves `fig_ablation_<DATASET>.png` and `fig_sparsity_<DATASET>.png`. Skips gracefully if the ablation / sparsity cells above were not run.

In [ ]:
# Ablation bar + sparsity curve (saved as PNG)
import matplotlib.pyplot as plt

if 'ABLATION_RESULTS' in globals() and ABLATION_RESULTS:
    names = list(ABLATION_RESULTS.keys())
    means = [ABLATION_RESULTS[n].mean() for n in names]
    stds  = [ABLATION_RESULTS[n].std()  for n in names]
    full_mae = ABLATION_RESULTS['Full V5 (all features)'].mean()
    colors = ['#B71C1C' if n.startswith('Full') else '#42A5F5' for n in names]
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.barh(range(len(names)), means, xerr=stds, color=colors, alpha=0.88, capsize=3)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    ax.axvline(full_mae, color='#B71C1C', ls='--', lw=1, alpha=0.6, label='Full V5')
    ax.invert_yaxis()
    ax.set_xlabel(f'MAE  ({DATASET_NAME})')
    ax.set_title('V5 ablation: leave-one-out (higher = component matters)', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'fig_ablation_{DATASET_NAME}.png', dpi=140, bbox_inches='tight')
    plt.show()
else:
    print("Run the ablation cell first.")

if 'SPARSITY_SWEEP' in globals() and SPARSITY_SWEEP:
    fmts = ['o-', 's--', '^:']
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for (k, d), fmt in zip(SPARSITY_SWEEP.items(), fmts):
        xs = sorted(d)
        ys = [d[sp].mean() for sp in xs]
        es = [d[sp].std()  for sp in xs]
        ax.errorbar([s*100 for s in xs], ys, yerr=es, fmt=fmt, capsize=3, label=k)
    ax.set_xlabel('Test sparsity  (% sensors blind)')
    ax.set_ylabel(f'MAE  ({DATASET_NAME})')
    ax.set_title('Sparsity sensitivity', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'fig_sparsity_{DATASET_NAME}.png', dpi=140, bbox_inches='tight')
    plt.show()
else:
    print("Run the sparsity cell first.")
